<center>
<img src="../images/fscampus_small2.png" width="1200"/>
</center>

<center>

# Investments

***Finance 2 - BFIN***

**Dr. Omer Cayirli**

Lecturer in Empirical Finance

omer_cayirli@uncbusiness.net
</center>

---



## Lecture 05
---

### Outline
* Portfolio Theory and Practice
  
  * Efficient Diversification
  
  * Portfolio Optimization

---



### Opportunity Set of Risky Assets
*   Minimum-variance frontier shows the risk-return opportunities available to the investor.
*   Efficient frontier
    
    *   There is an efficient portfolio for each risk level.
    
    *   Portfolios on the efficient frontier represent trade-offs in terms of risk and return.

For risky assets Alpha (A) and Beta (B):

| $E(r_A)$   | 8.00%    | $E(r_B)$   | 14.00%   | Cov($r_A$, $r_B$) | 0.002400 |
| :--------- | :------- | :--------- | :------- | :---------------- | :------- |
| $\sigma_A$ | 12.00%   | $\sigma_B$ | 20.00%   |                   |          |
| $\sigma_A^2$ | 0.014400 | $\sigma_B^2$ | 0.040000 |                   |          |

$$\quad\$$

| $w_A$ | $w_B$ | $E(r_p)$ | $\sigma_p^2$ | $\sigma_p$ |
| :---- | :---- | :------- | :----------- | :--------- |
| 100%  | 0%    | 8.00%    | 0.014400     | 12.00%     |
| 90%   | 10%   | 8.60%    | 0.012496     | 11.18%     |
| 80%   | 20%   | 9.20%    | 0.011584     | 10.76%     |
| 75.81% | 24.19% | 9.45%    | 0.011497     | 10.72%     |
| 70%   | 30%   | 9.80%    | 0.011664     | 10.80%     |
| 60%   | 40%   | 10.40%   | 0.012736     | 11.29%     |
| 50%   | 50%   | 11.00%   | 0.014800     | 12.17%     |
| 40%   | 60%   | 11.60%   | 0.017856     | 13.36%     |
| 30%   | 70%   | 12.20%   | 0.021904     | 14.80%     |
| 20%   | 80%   | 12.80%   | 0.026944     | 16.41%     |
| 10%   | 90%   | 13.40%   | 0.032976     | 18.16%     |
| 0%    | 100%  | 14.00%   | 0.040000     | 20.00%     |

---



In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- Input Parameter Widgets ---
style_desc = {'description_width': '200px'}
widget_layout_left = widgets.Layout(width='350px')
widget_layout_right = widgets.Layout(width='350px')

# Defaults match the Alpha/Beta example in cell 4.
er_a_widget = widgets.FloatText(value=0.080, step=0.0001, description='Expected Return Alpha (E(rA)):', style=style_desc, layout=widget_layout_left)
std_a_widget = widgets.FloatText(value=0.120, step=0.0001, description='Std. Dev. Alpha (σA):', style=style_desc, layout=widget_layout_left)
corr_ab_widget = widgets.FloatText(value=0.10, step=0.0001, description='Correlation (ρAB):', style=style_desc, layout=widget_layout_left)
cov_ab_widget = widgets.FloatText(value=0.0024, step=0.000001, description='Covariance (Cov(rA, rB)):', style=style_desc, layout=widget_layout_left)

er_b_widget = widgets.FloatText(value=0.140, step=0.0001, description='Expected Return Beta (E(rB)):', style=style_desc, layout=widget_layout_right)
std_b_widget = widgets.FloatText(value=0.200, step=0.0001, description='Std. Dev. Beta (σB):', style=style_desc, layout=widget_layout_right)

# --- Output Widgets ---
output_gmvp = widgets.Output()
output_table = widgets.Output()
output_plot = widgets.Output()

# --- Recalculate Button ---
recalculate_button = widgets.Button(
    description="Recalculate & Plot",
    button_style='info',
    tooltip='Click to update frontiers and table',
    icon='refresh',
    layout=widgets.Layout(width='auto', min_width='150px', margin='10px 0 0 0')
)

# --- Linking Logic for Correlation and Covariance ---
updating_internally = False

def update_covariance(change):
    global updating_internally
    if updating_internally:
        return
    updating_internally = True
    try:
        std_a = std_a_widget.value
        std_b = std_b_widget.value
        corr_ab = corr_ab_widget.value
        if np.all(np.isfinite([std_a, std_b, corr_ab])) and std_a >= 0 and std_b >= 0:
            # Keep full precision so valid boundary correlations remain valid.
            cov_ab_widget.value = corr_ab * std_a * std_b
    finally:
        updating_internally = False

def update_correlation(change):
    global updating_internally
    if updating_internally:
        return
    updating_internally = True
    try:
        std_a = std_a_widget.value
        std_b = std_b_widget.value
        cov_ab = cov_ab_widget.value
        if np.all(np.isfinite([std_a, std_b, cov_ab])) and std_a >= 0 and std_b >= 0:
            if std_a > 0 and std_b > 0:
                # Show the implied correlation; invalid inputs are rejected on calculation.
                corr_ab_widget.value = cov_ab / (std_a * std_b)
            else:
                corr_ab_widget.value = 0.0 if cov_ab == 0 else np.nan
    finally:
        updating_internally = False

def on_std_dev_change(change):
    update_covariance(None)

corr_ab_widget.observe(update_covariance, names='value')
cov_ab_widget.observe(update_correlation, names='value')
std_a_widget.observe(on_std_dev_change, names='value')
std_b_widget.observe(on_std_dev_change, names='value')

# --- Core Calculation and Plotting Logic ---
def calculate_and_display_frontiers(b=None):
    with output_gmvp: clear_output(wait=True)
    with output_table: clear_output(wait=True)
    with output_plot: clear_output(wait=True)
    try:
        er_a = er_a_widget.value; std_a = std_a_widget.value; var_a = std_a**2
        er_b = er_b_widget.value; std_b = std_b_widget.value; var_b = std_b**2
        cov_ab = cov_ab_widget.value

        if not np.all(np.isfinite([er_a, std_a, er_b, std_b, cov_ab])):
            raise ValueError("Enter finite numbers for returns, standard deviations, and covariance.")
        if std_a < 0 or std_b < 0:
            raise ValueError("Standard deviations must be nonnegative.")
        corr_ab = corr_ab_widget.value
        if not np.isfinite(corr_ab):
            raise ValueError("Enter a finite correlation; covariance must be zero if either standard deviation is zero.")
        if abs(corr_ab) > 1 and not np.isclose(abs(corr_ab), 1, rtol=1e-12, atol=0):
            raise ValueError("Correlation must be between -1 and 1.")
        covariance_limit = std_a * std_b
        if abs(cov_ab) > covariance_limit and not np.isclose(abs(cov_ab), covariance_limit, rtol=1e-12, atol=0):
            raise ValueError(f"Covariance must be between {-covariance_limit:.6g} and {covariance_limit:.6g}.")

        plot_weights_a = np.linspace(0, 1, 100)
        plot_portfolio_returns = np.array([w_a * er_a + (1 - w_a) * er_b for w_a in plot_weights_a])
        plot_portfolio_variances = np.array([(w_a**2 * var_a) + ((1 - w_a)**2 * var_b) + (2 * w_a * (1 - w_a) * cov_ab) for w_a in plot_weights_a])
        # Inputs are valid; clip only negligible floating-point roundoff below zero.
        plot_portfolio_variances = np.maximum(0, plot_portfolio_variances)
        plot_portfolio_std_devs = np.sqrt(plot_portfolio_variances)

        # This form avoids cancellation when the assets have almost identical risk.
        denominator_gmvp = (std_a - std_b)**2 + 2 * max(0, covariance_limit - cov_ab)
        constant_variance = denominator_gmvp == 0
        if constant_variance:
            # All allocations minimize variance; select the highest-return one.
            w_a_gmvp = 0.5 if er_a == er_b else float(er_a > er_b)
        else:
            w_a_gmvp = (var_b - cov_ab) / denominator_gmvp
        w_a_gmvp = np.clip(w_a_gmvp, 0, 1); w_b_gmvp = 1 - w_a_gmvp
        gmvp_return = w_a_gmvp * er_a + w_b_gmvp * er_b
        gmvp_variance = (w_a_gmvp**2 * var_a) + (w_b_gmvp**2 * var_b) + (2 * w_a_gmvp * w_b_gmvp * cov_ab)
        gmvp_variance = max(0, gmvp_variance)
        gmvp_std_dev = np.sqrt(gmvp_variance)

        with output_gmvp:
            display(widgets.HTML("<h4>Global Minimum Variance Portfolio (GMVP):</h4>"))
            gmvp_df = pd.DataFrame({
                'Weight Alpha (wA)': [f"{w_a_gmvp*100:.2f}%"], 'Weight Beta (wB)': [f"{w_b_gmvp*100:.2f}%"],
                'Expected Return E(Rp)': [f"{gmvp_return*100:.2f}%"], 'Variance (σp²)': [f"{gmvp_variance:.6f}"],
                'Std. Deviation (σp)': [f"{gmvp_std_dev*100:.2f}%"]})
            display(gmvp_df)

        table_weights_a = list(np.linspace(1, 0, 11))
        if not any(np.isclose(w_a_gmvp, w) for w in table_weights_a):
            table_weights_a.append(w_a_gmvp)
            table_weights_a.sort(reverse=True)
        table_data = []
        for w_a_t in table_weights_a:
            w_b_t = 1 - w_a_t
            rp_t = w_a_t * er_a + w_b_t * er_b
            var_p_t = (w_a_t**2 * var_a) + (w_b_t**2 * var_b) + (2 * w_a_t * w_b_t * cov_ab)
            var_p_t = max(0, var_p_t)
            std_p_t = np.sqrt(var_p_t)
            weight_precision = 2 if abs(w_a_t - w_a_gmvp) < 1e-10 else 0
            table_data.append([f"{w_a_t*100:.{weight_precision}f}%", f"{w_b_t*100:.{weight_precision}f}%", f"{rp_t*100:.2f}%", f"{var_p_t:.6f}", f"{std_p_t*100:.2f}%"])
        df_table = pd.DataFrame(table_data, columns=["wA", "wB", "E(rp)", "σp²", "σp"])
        with output_table:
            display(widgets.HTML("<h4>Portfolio Combinations Table:</h4>"))
            display(df_table)

        with output_plot:
            fig, ax = plt.subplots(figsize=(9, 6))
            ax.plot(plot_portfolio_std_devs, plot_portfolio_returns, linestyle='-', color='blue', label='Minimum Variance Frontier')
            
            if constant_variance or er_a == er_b:
                ax.plot([gmvp_std_dev], [gmvp_return], marker='o', color='green',
                        label='Efficient Portfolio (GMVP)', zorder=3)
            else:
                if er_a > er_b:
                    branch_weights = plot_weights_a[plot_weights_a > w_a_gmvp]
                else:
                    branch_weights = plot_weights_a[plot_weights_a < w_a_gmvp]
                efficient_weights = np.sort(np.r_[branch_weights, w_a_gmvp])
                efficient_returns = efficient_weights * er_a + (1 - efficient_weights) * er_b
                efficient_variances = (efficient_weights**2 * var_a
                                       + (1 - efficient_weights)**2 * var_b
                                       + 2 * efficient_weights * (1 - efficient_weights) * cov_ab)
                ax.plot(np.sqrt(np.maximum(0, efficient_variances)), efficient_returns,
                        color='green', linewidth=2.5, label='Efficient Frontier', zorder=3)

            ax.scatter([std_a], [er_a], color='red', marker='o', s=100, label='Alpha', zorder=4)
            ax.scatter([std_b], [er_b], color='purple', marker='o', s=100, label='Beta', zorder=4)
            ax.scatter([gmvp_std_dev], [gmvp_return], color='black', marker='*', s=200, label='GMVP', zorder=5)
            ax.set_title('Portfolio Risk: Minimum Variance & Efficient Frontier'); ax.set_xlabel('Portfolio Std Dev (σp)'); ax.set_ylabel('Portfolio Exp Return (E(rp))')
            ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y*100:.2f}%')); ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x*100:.2f}%'))
            ax.legend(); ax.grid(True, linestyle='--', alpha=0.7); plt.tight_layout(); plt.show()
    except Exception as e:
        with output_gmvp: clear_output(wait=True); print(f"Error in GMVP: {e}")
        with output_table: clear_output(wait=True); print(f"Error in Table: {e}")
        with output_plot: clear_output(wait=True); print(f"Error in Plot: {e}")

# --- Linking Button and Initial Display ---
recalculate_button.on_click(calculate_and_display_frontiers)

title_widget = widgets.HTML("<h2>Portfolio Risk Calculation (Alpha and Beta)</h2>", layout=widgets.Layout(width='100%'))
formula_cov_widget = widgets.HTML("For a two-asset portfolio (using Covariance):<br>    σ²(rp) = w₁²σ²(r₁) + w₂²σ²(r₂) + 2w₁w₂Cov(r₁, r₂)", layout=widgets.Layout(margin='0 0 2px 0'))
formula_corr_widget = widgets.HTML("For a two-asset portfolio (using Correlation):<br>    σ²(rp) = w₁²σ²(r₁) + w₂²σ²(r₂) + 2w₁w₂ρ₁₂σ₁σ₂", layout=widgets.Layout(margin='0 0 15px 0'))
inputs_col_a_corr_cov = widgets.VBox([er_a_widget, std_a_widget, corr_ab_widget, cov_ab_widget])
inputs_col_b = widgets.VBox([er_b_widget, std_b_widget], layout=widgets.Layout(margin='0 0 0 20px'))
left_panel_with_button = widgets.VBox([inputs_col_a_corr_cov, recalculate_button])
input_ui_grid = widgets.HBox([left_panel_with_button, inputs_col_b])

full_ui = widgets.VBox([
    title_widget, formula_cov_widget, formula_corr_widget, input_ui_grid,
    output_gmvp, widgets.HTML("<hr>"), output_table, widgets.HTML("<hr>"), output_plot
])

on_std_dev_change(None)
display(full_ui)
calculate_and_display_frontiers()

In [16]:
# If needed, uncomment and run once:
# %pip install -U yfinance curl_cffi pandas pandas-datareader numpy scipy matplotlib ipywidgets

import datetime
import warnings
import yfinance as yf
import pandas as pd
import numpy as np
import pandas_datareader.data as web
from scipy.optimize import minimize
from IPython.display import display, HTML, clear_output
import matplotlib.pyplot as plt
import ipywidgets as widgets

# --- 1. Data Download and Preparation ---
tickers = ['AMD', 'AMZN', 'F', 'JPM', 'NVDA', 'PFE', 'TGT', 'XOM']
default_start_date = datetime.date(2002, 12, 1)
last_month_end = (pd.Timestamp.today().normalize().replace(day=1) - pd.Timedelta(days=1)).date()

def run_opportunity_set(start_date, through_date):
    if start_date >= through_date:
        print('The start date must be earlier than the through-date.')
        return None

    end_date = through_date + datetime.timedelta(days=1)  # Yahoo end is exclusive.
    print(f'Downloading monthly adjusted-close data from {start_date} through {through_date}...')

    try:
        downloaded = yf.download(
            tickers, start=start_date, end=end_date, interval='1mo',
            group_by='ticker', progress=False, auto_adjust=True, actions=False
        )
        if downloaded.empty:
            raise ValueError('yf.download returned an empty DataFrame.')

        # With auto_adjust=True, Close is the adjusted-close series.
        series = []
        for ticker in tickers:
            if ticker in downloaded.columns.get_level_values(0):
                ticker_data = downloaded[ticker]
                if 'Close' in ticker_data.columns and ticker_data['Close'].notna().any():
                    series.append(ticker_data['Close'].rename(ticker))
        if not series:
            raise ValueError('No valid adjusted-close data was returned for any ticker.')

        adjusted_close = pd.concat(series, axis=1).sort_index().dropna(how='all')
        monthly_returns = adjusted_close.pct_change(fill_method=None).dropna(how='all')
        monthly_returns.index = pd.DatetimeIndex(monthly_returns.index).to_period('M').to_timestamp()
        min_observations = 12
        monthly_returns = monthly_returns.loc[:, monthly_returns.notna().sum() >= min_observations]
        if monthly_returns.shape[1] < 2:
            raise ValueError('At least two assets need 12 monthly returns each.')

        available = monthly_returns.columns.tolist()
        skipped = [ticker for ticker in tickers if ticker not in available]
        if skipped:
            print(f"Skipped unavailable or short-history tickers: {', '.join(skipped)}")

        with warnings.catch_warnings():
            warnings.simplefilter('ignore', FutureWarning)
            ff_factors = web.DataReader(
                'F-F_Research_Data_Factors', 'famafrench',
                start=start_date, end=through_date
            )[0]
        rf_series = (ff_factors['RF'].astype(float) / 100).rename('rf')
        rf_series.index = ff_factors.index.to_timestamp(how='start')

        # Optimization requires the same observed months for every asset.
        aligned = monthly_returns.join(rf_series, how='inner').dropna()
        if len(aligned) < min_observations:
            raise ValueError('Fewer than 12 complete months remain after aligning assets and RF.')
        asset_returns = aligned[available]
        rf_rate_avg = aligned['rf'].mean()
    except Exception as exc:
        print(f'Data Acquisition Error: {exc}')
        return None

    sample_start, sample_end = asset_returns.index.min(), asset_returns.index.max()
    print(f"Sample: {sample_start:%Y-%m} to {sample_end:%Y-%m} "
          f"({len(asset_returns)} months); assets: {', '.join(available)}")

    # --- 2. Portfolio Optimization ---
    mean_returns = asset_returns.mean()
    cov_matrix = asset_returns.cov()
    means = mean_returns.to_numpy()
    cov = cov_matrix.to_numpy()
    num_assets = len(available)
    ones = np.ones(num_assets)

    def portfolio_variance(weights):
        return float(weights @ cov @ weights)

    try:
        minimum_variance_weights = np.linalg.solve(cov, ones)
        w_short = minimum_variance_weights / (ones @ minimum_variance_weights)
        er_short = float(w_short @ means)
        std_short = np.sqrt(portfolio_variance(w_short))

        constraints_no_short = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
        bounds_no_short = [(0, 1)] * num_assets
        initial_guess = np.full(num_assets, 1 / num_assets)
        result_no_short = minimize(
            portfolio_variance, initial_guess, method='SLSQP',
            bounds=bounds_no_short, constraints=constraints_no_short,
            options={'ftol': 1e-12, 'maxiter': 1000}
        )
        if not result_no_short.success:
            raise RuntimeError(result_no_short.message)
        w_no_short = result_no_short.x
        er_no_short = float(w_no_short @ means)
        std_no_short = np.sqrt(portfolio_variance(w_no_short))
    except (np.linalg.LinAlgError, RuntimeError, ValueError) as exc:
        print(f'Portfolio Optimization Error: {exc}')
        return None

    sharpe_short = (er_short - rf_rate_avg) / std_short
    sharpe_no_short = (er_no_short - rf_rate_avg) / std_no_short

    return_range = mean_returns.max() - mean_returns.min()
    target_returns = np.unique(np.r_[
        np.linspace(mean_returns.min(), mean_returns.max() + 0.1 * return_range, 50),
        mean_returns.max()
    ])

    def solve_frontier(target_return, allow_short):
        constraints = [
            {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
            {'type': 'eq', 'fun': lambda w: w @ means - target_return}
        ]
        result = minimize(
            portfolio_variance, initial_guess, method='SLSQP',
            bounds=None if allow_short else bounds_no_short,
            constraints=constraints, options={'ftol': 1e-12, 'maxiter': 1000}
        )
        return np.sqrt(portfolio_variance(result.x)) if result.success else np.nan

    frontier_std_short = [solve_frontier(target, True) for target in target_returns]
    frontier_std_no_short = [solve_frontier(target, False) for target in target_returns]

    # --- 3. Assemble and Display ---
    results_df = pd.DataFrame({
        'Global MV Portfolios No Short': w_no_short,
        'Global MV Portfolios With Short': w_short
    }, index=available)
    summary_df = pd.DataFrame({
        'Global MV Portfolios No Short': [er_no_short, std_no_short, rf_rate_avg, sharpe_no_short],
        'Global MV Portfolios With Short': [er_short, std_short, rf_rate_avg, sharpe_short]
    }, index=['E(Rp)', 'σp', 'rf', 'Sharpe Ratio'])
    final_table = pd.concat([results_df, summary_df])

    display(HTML('<h3>Opportunity Set of Risky Assets</h3>'))
    display(final_table.style.format('{:.2%}').format(
        subset=pd.IndexSlice[['Sharpe Ratio'], :], formatter='{:.4f}'
    ))
    print('\n---')

    # --- 4. Generate the Plot ---
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.plot(frontier_std_short, target_returns, '--', color='gray',
            label='MV Frontier (Shorts Allowed)')
    ax.plot(frontier_std_no_short, target_returns, '-', color='blue',
            label='MV Frontier (No Shorts)')
    asset_risk = np.sqrt(np.diag(cov))
    ax.scatter(asset_risk, means, marker='o', s=60, label='Individual Assets')
    for ticker in available:
        ax.annotate(ticker, (np.sqrt(cov_matrix.loc[ticker, ticker]) * 1.01,
                             mean_returns.loc[ticker]))
    ax.scatter(std_short, er_short, marker='*', color='red', s=200,
               zorder=5, label='GMVP (Shorts Allowed)')
    ax.scatter(std_no_short, er_no_short, marker='*', color='darkorange', s=200,
               zorder=5, label='GMVP (No Shorts)')
    ax.set_title(f'Minimum Variance Frontiers ({sample_start:%Y-%m} to {sample_end:%Y-%m})',
                 fontsize=14)
    ax.set_xlabel('Monthly Standard Deviation (Risk)', fontsize=12)
    ax.set_ylabel('Mean Monthly Total Return', fontsize=12)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.2%}'))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.2%}'))
    ax.legend()
    ax.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.show()
    return final_table

start_box = widgets.Text(value=default_start_date.isoformat(), description='Start:')
through_box = widgets.Text(value=last_month_end.isoformat(), description='Through:')
run_button = widgets.Button(description='Download and run', button_style='primary')
output = widgets.Output()

def on_run(_):
    with output:
        clear_output(wait=True)
        try:
            start_date = datetime.date.fromisoformat(start_box.value.strip().replace('.', '-'))
            through_date = datetime.date.fromisoformat(through_box.value.strip().replace('.', '-'))
        except ValueError:
            print('Invalid date. Use YYYY-MM-DD or YYYY.MM.DD.')
            return
        run_opportunity_set(start_date, through_date)

run_button.on_click(on_run)
display(widgets.VBox([
    widgets.HBox([start_box, through_box]),
    run_button,
    output
]))


In [17]:
# If needed, uncomment and run once:
# %pip install -U yfinance curl_cffi pandas pandas-datareader numpy scipy matplotlib ipywidgets

import datetime
import warnings
import yfinance as yf
import pandas as pd
import numpy as np
import pandas_datareader.data as web
from scipy.optimize import minimize
from IPython.display import display, HTML, clear_output
import matplotlib.pyplot as plt
import ipywidgets as widgets

# --- 1. Analyze each sub-sample using its own monthly risk-free average ---
def analyze_subsample_period(period_data):
    returns = period_data.drop(columns='rf')
    risk_free = float(period_data['rf'].mean())
    mean_returns = returns.mean()
    cov_matrix = returns.cov()
    means = mean_returns.to_numpy()
    cov = cov_matrix.to_numpy()
    assets = returns.columns.tolist()
    num_assets = len(assets)
    initial_guess = np.full(num_assets, 1 / num_assets)

    def neg_sharpe_ratio(weights):
        expected_return = float(weights @ means)
        variance = float(weights @ cov @ weights)
        return -(expected_return - risk_free) / np.sqrt(variance) if variance > 1e-12 else 1e9

    constraint = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
    options = {'ftol': 1e-12, 'maxiter': 1000}
    result_short = minimize(
        neg_sharpe_ratio, initial_guess, method='SLSQP',
        constraints=constraint, options=options
    )
    bounds_no_short = [(0, 1)] * num_assets
    result_no_short = minimize(
        neg_sharpe_ratio, initial_guess, method='SLSQP',
        bounds=bounds_no_short, constraints=constraint, options=options
    )
    if not result_short.success or not result_no_short.success:
        raise RuntimeError(
            f'Sharpe optimization failed: shorts={result_short.message}; '
            f'no shorts={result_no_short.message}'
        )

    w_s, w_ns = result_short.x, result_no_short.x
    er_s, er_ns = float(w_s @ means), float(w_ns @ means)
    std_s = np.sqrt(float(w_s @ cov @ w_s))
    std_ns = np.sqrt(float(w_ns @ cov @ w_ns))
    sharpe_s = (er_s - risk_free) / std_s
    sharpe_ns = (er_ns - risk_free) / std_ns

    def portfolio_variance(weights):
        return float(weights @ cov @ weights)

    lower = min(mean_returns.min(), er_ns, er_s)
    upper = max(mean_returns.max(), er_ns, er_s)
    target_returns = np.unique(np.r_[np.linspace(lower, upper + 0.1 * (upper - lower), 50), mean_returns.max()])

    def solve_frontier(target_return, allow_short):
        constraints = [
            {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
            {'type': 'eq', 'fun': lambda w: w @ means - target_return}
        ]
        result = minimize(
            portfolio_variance, initial_guess, method='SLSQP',
            bounds=None if allow_short else bounds_no_short,
            constraints=constraints, options=options
        )
        return np.sqrt(portfolio_variance(result.x)) if result.success else np.nan

    frontier_s = [solve_frontier(target, True) for target in target_returns]
    frontier_ns = [solve_frontier(target, False) for target in target_returns]
    return {
        'w_s': w_s, 'er_s': er_s, 'std_s': std_s, 'sharpe_s': sharpe_s,
        'w_ns': w_ns, 'er_ns': er_ns, 'std_ns': std_ns, 'sharpe_ns': sharpe_ns,
        'frontier_s': frontier_s, 'frontier_ns': frontier_ns,
        'target_returns': target_returns, 'mean_returns': mean_returns,
        'cov_matrix': cov_matrix, 'risk_free': risk_free, 'assets': assets,
        'sample_start': period_data.index.min(), 'sample_end': period_data.index.max(),
        'months': len(period_data)
    }

# --- 2. Download adjusted closes and form comparable sub-samples ---
subsample_tickers = ['AMD', 'AMZN', 'F', 'JPM', 'NVDA', 'PFE', 'TGT', 'XOM']
subsample_default_start_date = datetime.date(2002, 12, 1)
subsample_default_split_date = datetime.date(2012, 12, 31)  # January 2013 begins sub-sample 2.
subsample_last_month_end = (pd.Timestamp.today().normalize().replace(day=1) - pd.Timedelta(days=1)).date()

def run_subsample_analysis(start_date, split_date, through_date):
    if not start_date < split_date < through_date:
        print('The split date must fall between the start and through dates.')
        return None

    end_date = through_date + datetime.timedelta(days=1)  # Yahoo end is exclusive.
    print(f'Downloading monthly adjusted-close data from {start_date} through {through_date}...')
    try:
        downloaded = yf.download(
            subsample_tickers, start=start_date, end=end_date, interval='1mo',
            group_by='ticker', progress=False, auto_adjust=True, actions=False
        )
        if downloaded.empty:
            raise ValueError('yf.download returned an empty DataFrame.')

        # With auto_adjust=True, Close is the adjusted-close series.
        series = []
        for ticker in subsample_tickers:
            if ticker in downloaded.columns.get_level_values(0):
                ticker_data = downloaded[ticker]
                if 'Close' in ticker_data.columns and ticker_data['Close'].notna().any():
                    series.append(ticker_data['Close'].rename(ticker))
        if not series:
            raise ValueError('No valid adjusted-close data was returned for any ticker.')

        adjusted_close = pd.concat(series, axis=1).sort_index().dropna(how='all')
        monthly_returns = adjusted_close.pct_change(fill_method=None).dropna(how='all')
        monthly_returns.index = pd.DatetimeIndex(monthly_returns.index).to_period('M').to_timestamp()

        with warnings.catch_warnings():
            warnings.simplefilter('ignore', FutureWarning)
            ff_factors = web.DataReader(
                'F-F_Research_Data_Factors', 'famafrench',
                start=start_date, end=through_date
            )[0]
        rf_series = (ff_factors['RF'].astype(float) / 100).rename('rf')
        rf_series.index = ff_factors.index.to_timestamp(how='start')
        aligned = monthly_returns.join(rf_series, how='inner')
        aligned = aligned.loc[aligned['rf'].notna()]

        first = aligned.loc[
            (aligned.index >= pd.Timestamp(start_date)) &
            (aligned.index <= pd.Timestamp(split_date))
        ]
        second = aligned.loc[
            (aligned.index > pd.Timestamp(split_date)) &
            (aligned.index <= pd.Timestamp(through_date))
        ]
        min_observations = 12
        common_assets = [
            ticker for ticker in subsample_tickers
            if ticker in aligned.columns
            and first[ticker].notna().sum() >= min_observations
            and second[ticker].notna().sum() >= min_observations
        ]
        if len(common_assets) < 2:
            raise ValueError('At least two assets need 12 monthly returns in each sub-sample.')

        first = first[common_assets + ['rf']].dropna()
        second = second[common_assets + ['rf']].dropna()
        if len(first) < min_observations or len(second) < min_observations:
            raise ValueError('Fewer than 12 complete months remain in a sub-sample.')

        skipped = [ticker for ticker in subsample_tickers if ticker not in common_assets]
        if skipped:
            print(f"Skipped unavailable or short-history tickers: {', '.join(skipped)}")
    except Exception as exc:
        print(f'Data Acquisition Error: {exc}')
        return None

    print(f"First sub-sample: {first.index.min():%Y-%m} to {first.index.max():%Y-%m} "
          f"({len(first)} months); second: {second.index.min():%Y-%m} to "
          f"{second.index.max():%Y-%m} ({len(second)} months)")
    try:
        results = [analyze_subsample_period(first), analyze_subsample_period(second)]
    except (RuntimeError, ValueError, np.linalg.LinAlgError) as exc:
        print(f'Portfolio Optimization Error: {exc}')
        return None

    # --- 3. Side-by-side tables ---
    with subsample_output_tables:
        html_tables = ''
        for i, result in enumerate(results, start=1):
            weights = pd.DataFrame({
                'No Short': result['w_ns'], 'With Short': result['w_s']
            }, index=result['assets'])
            summary = pd.DataFrame({
                'No Short': [result['er_ns'], result['std_ns'], result['risk_free'], result['sharpe_ns']],
                'With Short': [result['er_s'], result['std_s'], result['risk_free'], result['sharpe_s']]
            }, index=['E(rp)', 'σp', 'rf', 'Sharpe Ratio'])
            table = pd.concat([weights, summary])
            styled = table.style.format('{:.2%}').format(
                subset=pd.IndexSlice[['Sharpe Ratio'], :], formatter='{:.4f}'
            )
            label = f"Sub-sample {i}: {result['sample_start']:%Y-%m} to {result['sample_end']:%Y-%m}"
            html_tables += f"<div style='flex:1; padding:0 10px;'><h4>{label}</h4>{styled.to_html()}</div>"
        display(HTML(f"<h3>Optimal Risky Portfolios</h3><div style='display:flex;'>{html_tables}</div><hr>"))

    # --- 4. Comparable plots ---
    with subsample_output_plots:
        fig, axes = plt.subplots(1, 2, figsize=(18, 7))
        x_values, y_values = [], []
        for result in results:
            x_values.extend(np.sqrt(np.diag(result['cov_matrix'])))
            x_values.extend([result['std_s'], result['std_ns']])
            x_values.extend(np.asarray(result['frontier_s'])[np.isfinite(result['frontier_s'])])
            x_values.extend(np.asarray(result['frontier_ns'])[np.isfinite(result['frontier_ns'])])
            y_values.extend(result['target_returns'])
            y_values.extend(result['mean_returns'])
            y_values.extend([result['er_s'], result['er_ns']])
        x_max = max(x_values) * 1.1
        y_min, y_max = min(y_values), max(y_values)
        y_pad = max((y_max - y_min) * 0.05, 0.001)

        for i, (ax, result) in enumerate(zip(axes, results), start=1):
            ax.plot(result['frontier_s'], result['target_returns'], '--',
                    color='gray', label='MVF (Shorts)')
            ax.plot(result['frontier_ns'], result['target_returns'], '-',
                    color='blue', label='MVF (No Shorts)')
            asset_risk = np.sqrt(np.diag(result['cov_matrix']))
            ax.scatter(asset_risk, result['mean_returns'], marker='o', s=60, label='Assets')
            ax.scatter(result['std_s'], result['er_s'], marker='*', color='red',
                       s=200, zorder=5, label='Optimal (Shorts)')
            ax.scatter(result['std_ns'], result['er_ns'], marker='*', color='orange',
                       s=200, zorder=5, label='Optimal (No Shorts)')
            for ticker in result['assets']:
                ax.annotate(ticker, (
                    np.sqrt(result['cov_matrix'].loc[ticker, ticker]) * 1.02,
                    result['mean_returns'].loc[ticker]
                ))
            ax.set_title(
                f"Sub-sample {i} ({result['sample_start']:%Y-%m} to "
                f"{result['sample_end']:%Y-%m})", fontsize=14
            )
            ax.set_xlabel('Monthly Standard Deviation (Risk)')
            ax.set_ylabel('Mean Monthly Total Return')
            ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.2%}'))
            ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.2%}'))
            ax.legend()
            ax.grid(True, linestyle=':', alpha=0.6)
            ax.set_xlim(left=0, right=x_max)
            ax.set_ylim(bottom=y_min - y_pad, top=y_max + y_pad)
        plt.tight_layout()
        plt.show()
    return results

subsample_start_box = widgets.Text(value=subsample_default_start_date.isoformat(), description='Start:')
subsample_split_box = widgets.Text(value=subsample_default_split_date.isoformat(), description='Split:')
subsample_through_box = widgets.Text(value=subsample_last_month_end.isoformat(), description='Through:')
subsample_run_button = widgets.Button(description='Download and run', button_style='primary')
subsample_output_tables, subsample_output_plots = widgets.Output(), widgets.Output()

def on_subsample_run(_):
    with subsample_output_tables:
        clear_output(wait=True)
    with subsample_output_plots:
        clear_output(wait=True)
    try:
        start_date = datetime.date.fromisoformat(subsample_start_box.value.strip().replace('.', '-'))
        split_date = datetime.date.fromisoformat(subsample_split_box.value.strip().replace('.', '-'))
        through_date = datetime.date.fromisoformat(subsample_through_box.value.strip().replace('.', '-'))
    except ValueError:
        with subsample_output_tables:
            print('Invalid date. Use YYYY-MM-DD or YYYY.MM.DD.')
        return
    with subsample_output_tables:
        run_subsample_analysis(start_date, split_date, through_date)

subsample_run_button.on_click(on_subsample_run)
display(widgets.VBox([
    widgets.HBox([subsample_start_box, subsample_split_box, subsample_through_box]),
    subsample_run_button,
    subsample_output_tables,
    subsample_output_plots
]))


---

### Capital Allocation to Risk-free and Risky Assets
* Complete portfolio consists of the risky asset and the risk-free asset.
  
  * A portfolio risk closer to the desired level can be achieved by changing the weights of risk-free asset and risky assets in the complete portfolio.

$W_{rf} + W_{risky} = 1$

$W_{rf} = 1 - W_{risky}$

$E(r_p) = (1 - W_{risky}) \cdot r_f + W_{risky} \cdot E(r_{risky})$

$E(r_p) = r_f + W_{risky} \cdot [E(r_{risky}) - r_f]$

$\sigma^2(r_p) = w_{rf}^2 \cdot \sigma^2(r_f) + W_{risky}^2 \cdot \sigma^2(r_{risky}) + 2 \cdot W_{rf}W_{risky}Cov(r_f, r_{risky})$

Since $\sigma^2(r_f) = 0$ and $Cov(r_f, r_{risky}) = 0$,

$\sigma^2(r_p) = W_{risky}^2 \cdot \sigma^2(r_{risky})$

---



### Capital Allocation to Risk-free and Risky Assets
If $\sigma^2(r_p) = w_{risky}^2 \cdot \sigma^2(r_{risky})$ then $\sigma(r_p) = W_{risky} \cdot \sigma(r_{risky})$

Thus, $W_{risky} = \frac{\sigma(r_p)}{\sigma(r_{risky})}$

$E(r_p) = r_f + W_{risky} \cdot [E(r_{risky}) - r_f]$

$E(r_p) = r_f + \frac{\sigma(r_p)}{\sigma(r_{risky})} \cdot [E(r_{risky}) - r_f]$

$E(r_p) = r_f + \left[ \frac{E(r_{risky})-r_f}{\sigma(r_{risky})} \right] \cdot \sigma(r_p)$




---



### Capital Allocation to Risk-free and Risky Assets
* Suppose the risk-free interest rate is 5%, the expected return and the standard deviation of the risky portfolio are 12% and 15%, respectively. Then;

$E(r_p) = r_f + \left[ \frac{E(r_{risky}) - r_f}{\sigma(r_{risky})} \right] \cdot \sigma(r_p)$

$E(r_p) = 0.05 + \left[ \frac{0.12-0.05}{0.15} \right] \cdot \sigma(r_p)$

$E(r_p) = 0.05 + 0.4667 \cdot \sigma(r_p)$

<img src = "../images/slide_5/cal_example.svg">

---



In [6]:
# If needed, uncomment and run once:
# %pip install -U yfinance curl_cffi pandas pandas-datareader numpy scipy matplotlib ipywidgets

import yfinance as yf
import pandas as pd
import numpy as np
import pandas_datareader.data as web
from scipy.optimize import minimize
from IPython.display import display, HTML, clear_output
import matplotlib.pyplot as plt
import ipywidgets as widgets
import datetime
import warnings

# --- 1. Core Analysis Function ---
def analyze_cml_period(returns_df, rf_rate):
    """
    Analyze portfolio using only assets present in returns_df.
    No hidden fallbacks - raises on any failure.
    """
    returns_df = returns_df.dropna()  # Common intersection - no partial NaNs
    
    if returns_df.empty or len(returns_df) < 12:
        raise ValueError("Insufficient data: need at least 12 complete monthly returns")
    
    assets = returns_df.columns.tolist()
    mean_returns = returns_df.mean()
    cov_matrix = returns_df.cov()
    
    # FIX: Use arrays consistently
    mu = mean_returns.values
    Sigma = cov_matrix.values
    
    # FIX: Check singular once and reuse
    inv_cov = np.linalg.inv(Sigma)
    
    initial_guess = np.full(len(assets), 1.0 / len(assets))
    
    def port_var(w):
        return float(w @ Sigma @ w)
    
    def neg_sharpe_ratio(w, mean_ret, cov, rf):
        er = float(w @ mean_ret)
        var = float(w @ cov @ w)
        std = np.sqrt(var)
        # FIX: Barrier instead of raising (SLSQP calls on infeasible points)
        if std <= 1e-12:
            return 1e12
        return -(er - rf) / std

    constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
    bounds_no_short = tuple((0, 1) for _ in range(len(assets)))
    optimizer_options = {'ftol': 1e-12, 'maxiter': 1000}
    
    # --- GMVP (No Short) ---
    res_gmvp_ns = minimize(
        port_var, initial_guess,
        method='SLSQP', bounds=bounds_no_short, constraints=constraints, options=optimizer_options
    )
    if not res_gmvp_ns.success:
        raise RuntimeError(f"GMVP (No Short) optimization failed: {res_gmvp_ns.message}")
    w_gmvp_ns = res_gmvp_ns.x
    
    # --- GMVP (Short) - Analytical using inv_cov we already computed ---
    ones = np.ones(len(assets))
    w_gmvp_s = inv_cov @ ones / (ones @ inv_cov @ ones)
    if not np.isclose(w_gmvp_s.sum(), 1.0, atol=1e-6):
        raise ValueError(f"GMVP (Short) weights sum to {w_gmvp_s.sum()}, not 1.0")

    # --- Optimal Portfolios ---
    res_opt_ns = minimize(
        neg_sharpe_ratio, initial_guess,
        args=(mu, Sigma, rf_rate),
        method='SLSQP', bounds=bounds_no_short, constraints=constraints, options=optimizer_options
    )
    if not res_opt_ns.success:
        raise RuntimeError(f"Optimal (No Short) optimization failed: {res_opt_ns.message}")
    w_opt_ns = res_opt_ns.x
    
    # Optimal Sharpe (Short) - fully unconstrained except sum=1
    res_opt_s = minimize(
        neg_sharpe_ratio, initial_guess,
        args=(mu, Sigma, rf_rate),
        method='SLSQP', constraints=constraints, options=optimizer_options
    )
    if not res_opt_s.success:
        raise RuntimeError(f"Optimal (Short) optimization failed: {res_opt_s.message}")
    w_opt_s = res_opt_s.x
    
    def stats_of(w):
        er = float(w @ mu)
        sd = float(np.sqrt(w @ Sigma @ w))
        if sd <= 1e-12:
            raise ValueError("Degenerate portfolio (std≈0).")
        return {'w': w, 'er': er, 'std': sd, 'sharpe': (er - rf_rate) / sd}
    
    stats = {
        'gmvp_ns': stats_of(w_gmvp_ns),
        'gmvp_s': stats_of(w_gmvp_s),
        'opt_ns': stats_of(w_opt_ns),
        'opt_s': stats_of(w_opt_s),
    }
    
    lo = min(mean_returns.min(), stats['gmvp_ns']['er'])
    hi = max(mean_returns.max(), stats['opt_ns']['er'], stats['opt_s']['er'])
    target_returns_range = np.unique(np.r_[
        np.linspace(lo, hi, 50), stats['gmvp_ns']['er'], mean_returns.max()
    ])
    
    def solve_frontier(target_ret, allow_short):
        cons = [
            {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
            {'type': 'eq', 'fun': lambda w, trg=target_ret: float(w @ mu) - trg},
        ]
        bnds = None if allow_short else bounds_no_short
        res = minimize(port_var, initial_guess, method='SLSQP', bounds=bnds, constraints=cons, options=optimizer_options)
        return float(np.sqrt(res.fun)) if res.success else np.nan
    
    stats['frontier_s'] = [solve_frontier(tr, True) for tr in target_returns_range]
    stats['frontier_ns'] = [solve_frontier(tr, False) for tr in target_returns_range]
    
    return {
        'stats': stats,
        'mean_returns': mean_returns,
        'cov_matrix': cov_matrix,
        'rf_rate': rf_rate,
        'assets': assets,
        'target_returns': target_returns_range
    }

# --- 2. UI and Main Handler ---
cml_tickers = ['AMD', 'AMZN', 'F', 'JPM', 'NVDA', 'PFE', 'TGT', 'XOM']
cml_default_start_date = datetime.date(2002, 12, 1)
cml_last_month_end = (pd.Timestamp.today().normalize().replace(day=1) - pd.Timedelta(days=1)).date()
cml_start_box = widgets.Text(value=cml_default_start_date.isoformat(), description='Start:')
cml_through_box = widgets.Text(value=cml_last_month_end.isoformat(), description='Through:')
cml_run_button = widgets.Button(description='Download and run', button_style='primary')
cml_output_table, cml_output_plot = widgets.Output(), widgets.Output()

def run_cml_analysis(start, through):
    if start >= through:
        print('The start date must be earlier than the through-date.')
        return None

    end_date = through + datetime.timedelta(days=1)  # Yahoo end is exclusive.
    print(f'Downloading monthly adjusted-close data from {start} through {through}...')
    try:
        downloaded = yf.download(
            cml_tickers, start=start, end=end_date, interval='1mo',
            group_by='ticker', progress=False, auto_adjust=True, actions=False
        )
        if downloaded.empty:
            raise ValueError('yf.download returned an empty DataFrame.')

        # With auto_adjust=True, Close is the adjusted-close series.
        series = []
        for ticker in cml_tickers:
            if ticker in downloaded.columns.get_level_values(0):
                ticker_data = downloaded[ticker]
                if 'Close' in ticker_data.columns and ticker_data['Close'].notna().any():
                    series.append(ticker_data['Close'].rename(ticker))
        if not series:
            raise ValueError('No valid adjusted-close data was returned for any ticker.')

        adjusted_close = pd.concat(series, axis=1).sort_index().dropna(how='all')
        monthly_returns = adjusted_close.pct_change(fill_method=None).dropna(how='all')
        monthly_returns.index = pd.DatetimeIndex(monthly_returns.index).to_period('M').to_timestamp()
        min_observations = 12
        monthly_returns = monthly_returns.loc[:, monthly_returns.notna().sum() >= min_observations]
        if monthly_returns.shape[1] < 2:
            raise ValueError('At least two assets need 12 monthly returns each.')

        available = monthly_returns.columns.tolist()
        skipped = [ticker for ticker in cml_tickers if ticker not in available]
        if skipped:
            print(f"Skipped unavailable or short-history tickers: {', '.join(skipped)}")

        with warnings.catch_warnings():
            warnings.simplefilter('ignore', FutureWarning)
            ff_factors = web.DataReader(
                'F-F_Research_Data_Factors', 'famafrench', start=start, end=through
            )[0]
        rf_series = (ff_factors['RF'].astype(float) / 100).rename('rf')
        rf_series.index = ff_factors.index.to_timestamp(how='start')

        # Use complete, matching months for every asset and the risk-free series.
        aligned = monthly_returns.join(rf_series, how='inner').dropna()
        if len(aligned) < min_observations:
            raise ValueError('Fewer than 12 complete months remain after aligning assets and RF.')
        monthly_returns = aligned[available]
        rf_rate = float(aligned['rf'].mean())
    except Exception as exc:
        print(f'Data Acquisition Error: {exc}')
        return None

    try:
        results = analyze_cml_period(monthly_returns, rf_rate)
    except Exception as exc:
        print(f'Portfolio Analysis Error: {exc}')
        return None

    results['sample_start'] = monthly_returns.index.min()
    results['sample_end'] = monthly_returns.index.max()
    print(f"Sample: {results['sample_start']:%Y-%m} to {results['sample_end']:%Y-%m} "
          f"({len(monthly_returns)} months); average monthly RF: {rf_rate:.2%}")

    with cml_output_table:
        r = results['stats']
        assets = results['assets']
        
        port_cols = {
            'Global MV (No Short)': r['gmvp_ns']['w'],
            'Global MV (Short)': r['gmvp_s']['w'],
            'Optimal (No Short)': r['opt_ns']['w'],
            'Optimal (Short)': r['opt_s']['w']
        }
        res_df = pd.DataFrame(port_cols, index=assets)
        
        summary_data = {
            'Global MV (No Short)': [
                r['gmvp_ns']['er'], r['gmvp_ns']['std'],
                results['rf_rate'], r['gmvp_ns']['sharpe']
            ],
            'Global MV (Short)': [
                r['gmvp_s']['er'], r['gmvp_s']['std'],
                results['rf_rate'], r['gmvp_s']['sharpe']
            ],
            'Optimal (No Short)': [
                r['opt_ns']['er'], r['opt_ns']['std'],
                results['rf_rate'], r['opt_ns']['sharpe']
            ],
            'Optimal (Short)': [
                r['opt_s']['er'], r['opt_s']['std'],
                results['rf_rate'], r['opt_s']['sharpe']
            ]
        }
        summary_df = pd.DataFrame(
            summary_data,
            index=['E(rp)', 'σp', 'rf', 'Sharpe Ratio']
        )
        final_table = pd.concat([res_df, summary_df])
        styler = final_table.style.format('{:.2%}').format(
            subset=pd.IndexSlice[['Sharpe Ratio'], :], formatter="{:.4f}"
        )
        display(HTML("<h3>Opportunity Set of Risky Assets</h3>"), styler)
        print(f"\nAssets used: {', '.join(assets)}")
        print("---")

    with cml_output_plot:
        fig, ax = plt.subplots(figsize=(12, 8))
        r = results['stats']
        
        ax.plot(
            r['frontier_s'], results['target_returns'],
            linestyle='--', c='gray', label='MV Frontier (Shorts)'
        )
        ax.plot(
            r['frontier_ns'], results['target_returns'],
            linestyle='-', c='lightsteelblue', linewidth=1.5, label='MV Frontier (No Shorts)'
        )
        efficient_ns = np.where(
            results['target_returns'] >= r['gmvp_ns']['er'] - 1e-9,
            r['frontier_ns'], np.nan
        )
        ax.plot(efficient_ns, results['target_returns'], c='blue', linewidth=2.5,
                label='Efficient Frontier (No Shorts)')
        
        # FIX: Extract diagonal from .values
        asset_risk = np.sqrt(np.diag(results['cov_matrix'].values))
        ax.scatter(
            asset_risk, results['mean_returns'],
            marker='o', s=60, label='Individual Assets'
        )
        
        for ticker in results['assets']:
            ticker_std = np.sqrt(results['cov_matrix'].loc[ticker, ticker])
            ticker_return = results['mean_returns'].loc[ticker]
            ax.annotate(ticker, (ticker_std * 1.02, ticker_return))
        
        port_points = {
            'GMVP (No Shorts)': (r['gmvp_ns']['std'], r['gmvp_ns']['er']),
            'GMVP (Shorts)': (r['gmvp_s']['std'], r['gmvp_s']['er']),
            'Optimal (No Shorts)': (r['opt_ns']['std'], r['opt_ns']['er']),
            'Optimal (Shorts)': (r['opt_s']['std'], r['opt_s']['er'])
        }
        colors = ['darkorange', 'red', 'purple', 'green']
        for (name, (std, er)), color in zip(port_points.items(), colors):
            ax.scatter(std, er, marker='*', s=250, zorder=5, label=name, color=color)
        
        max_risk_for_cml = ax.get_xlim()[1]
        cml_ns_sharpe = r['opt_ns']['sharpe']
        cml_s_sharpe = r['opt_s']['sharpe']
        cml_x = np.array([0, max_risk_for_cml])
        ax.plot(
            cml_x, results['rf_rate'] + cml_x * cml_ns_sharpe,
            color='purple', linestyle='--', label='CAL (No Shorts)'
        )
        ax.plot(
            cml_x, results['rf_rate'] + cml_x * cml_s_sharpe,
            color='green', linestyle=':', label='CAL (Shorts)'
        )
        
        ax.set_title(
            f"Minimum Variance Frontiers, Optimal Portfolios, and CALs ({results['sample_start']:%Y-%m} to {results['sample_end']:%Y-%m})",
            fontsize=14
        )
        ax.set_xlabel('Monthly Standard Deviation (Risk)')
        ax.set_ylabel('Mean Monthly Total Return')
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.2%}'))
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.2%}'))
        ax.legend()
        ax.grid(True, linestyle=':', alpha=0.6)
        ax.set_xlim(left=0)
        plotted_returns = np.r_[
            results['target_returns'], results['mean_returns'].values,
            [results['rf_rate']],
            [point['er'] for point in r.values()
             if isinstance(point, dict) and 'er' in point]
        ]
        finite_returns = plotted_returns[np.isfinite(plotted_returns)]
        y_pad = max(np.ptp(finite_returns) * 0.05, 0.001)
        ax.set_ylim(bottom=min(0, finite_returns.min() - y_pad))
        plt.tight_layout()
        plt.show()

    return results

def on_cml_run(_):
    with cml_output_table:
        clear_output(wait=True)
    with cml_output_plot:
        clear_output(wait=True)
    try:
        start = datetime.date.fromisoformat(cml_start_box.value.strip().replace('.', '-'))
        through = datetime.date.fromisoformat(cml_through_box.value.strip().replace('.', '-'))
    except ValueError:
        with cml_output_table:
            print('Invalid date. Use YYYY-MM-DD or YYYY.MM.DD.')
        return
    with cml_output_table:
        run_cml_analysis(start, through)

cml_run_button.on_click(on_cml_run)
display(widgets.VBox([
    widgets.HBox([cml_start_box, cml_through_box]),
    cml_run_button,
    cml_output_table,
    cml_output_plot
]))


In [7]:
# If needed, uncomment and run once:
# %pip install -U yfinance curl_cffi pandas pandas-datareader numpy scipy matplotlib ipywidgets

import yfinance as yf
import pandas as pd
import numpy as np
import pandas_datareader.data as web
from scipy.optimize import minimize
from IPython.display import display, HTML, clear_output
import matplotlib.pyplot as plt
import ipywidgets as widgets
import datetime
import warnings

# --- 1. Core Analysis Function ---
def analyze_comparison_period(returns_df, rf_rate):
    """
    Analyze portfolio - no hidden fallbacks.
    """
    returns_df = returns_df.dropna()  # Common intersection
    
    if returns_df.empty or len(returns_df) < 12 or len(returns_df.columns) < 2:
        raise ValueError("Insufficient data: need at least 2 assets with 12 complete monthly returns")
    
    assets = returns_df.columns.tolist()
    mean_returns = returns_df.mean()
    cov_matrix = returns_df.cov()
    
    # Use arrays consistently
    mu = mean_returns.values
    Sigma = cov_matrix.values
    
    # Check singular once and reuse
    inv_cov = np.linalg.inv(Sigma)
    
    initial_guess = np.full(len(assets), 1.0 / len(assets))
    
    def port_var(w):
        return float(w @ Sigma @ w)
    
    def neg_sharpe_ratio(w, mean_ret, cov, rf):
        er = float(w @ mean_ret)
        var = float(w @ cov @ w)
        std = np.sqrt(var)
        # Barrier instead of raising (SLSQP calls on infeasible points)
        if std <= 1e-12:
            return 1e12
        return -(er - rf) / std

    constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
    bounds_no_short = tuple((0, 1) for _ in range(len(assets)))
    optimizer_options = {'ftol': 1e-12, 'maxiter': 1000}
    
    # --- GMVP (No Short) ---
    res_gmvp_ns = minimize(
        port_var, initial_guess,
        method='SLSQP', bounds=bounds_no_short, constraints=constraints, options=optimizer_options
    )
    if not res_gmvp_ns.success:
        raise RuntimeError(f"GMVP (No Short) optimization failed: {res_gmvp_ns.message}")
    w_gmvp_ns = res_gmvp_ns.x
    
    # --- GMVP (Short) - Analytical using inv_cov we already computed ---
    ones = np.ones(len(assets))
    w_gmvp_s = inv_cov @ ones / (ones @ inv_cov @ ones)
    if not np.isclose(w_gmvp_s.sum(), 1.0, atol=1e-6):
        raise ValueError(f"GMVP (Short) weights sum to {w_gmvp_s.sum()}, not 1.0")

    # --- Optimal Portfolios ---
    res_opt_ns = minimize(
        neg_sharpe_ratio, initial_guess,
        args=(mu, Sigma, rf_rate),
        method='SLSQP', bounds=bounds_no_short, constraints=constraints, options=optimizer_options
    )
    if not res_opt_ns.success:
        raise RuntimeError(f"Optimal (No Short) optimization failed: {res_opt_ns.message}")
    w_opt_ns = res_opt_ns.x
    
    # Optimal Sharpe (Short) - fully unconstrained except sum=1
    res_opt_s = minimize(
        neg_sharpe_ratio, initial_guess,
        args=(mu, Sigma, rf_rate),
        method='SLSQP', constraints=constraints, options=optimizer_options
    )
    if not res_opt_s.success:
        raise RuntimeError(f"Optimal (Short) optimization failed: {res_opt_s.message}")
    w_opt_s = res_opt_s.x
    
    def stats_of(w):
        er = float(w @ mu)
        sd = float(np.sqrt(w @ Sigma @ w))
        if sd <= 1e-12:
            raise ValueError("Degenerate portfolio (std≈0).")
        return {'w': w, 'er': er, 'std': sd, 'sharpe': (er - rf_rate) / sd}
    
    stats = {
        'gmvp_ns': stats_of(w_gmvp_ns),
        'gmvp_s': stats_of(w_gmvp_s),
        'opt_ns': stats_of(w_opt_ns),
        'opt_s': stats_of(w_opt_s),
    }
    
    lo = min(mean_returns.min(), stats['gmvp_ns']['er'])
    hi = max(mean_returns.max(), stats['opt_ns']['er'], stats['opt_s']['er'])
    target_returns_range = np.unique(np.r_[
        np.linspace(lo, hi, 50), stats['gmvp_ns']['er'], mean_returns.max()
    ])
    
    def solve_frontier(target_ret, allow_short):
        cons = [
            {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
            {'type': 'eq', 'fun': lambda w, trg=target_ret: float(w @ mu) - trg},
        ]
        bnds = None if allow_short else bounds_no_short
        res = minimize(port_var, initial_guess, method='SLSQP', bounds=bnds, constraints=cons, options=optimizer_options)
        return float(np.sqrt(res.fun)) if res.success else np.nan
    
    stats['frontier_s'] = [solve_frontier(tr, True) for tr in target_returns_range]
    stats['frontier_ns'] = [solve_frontier(tr, False) for tr in target_returns_range]
    
    return {
        'stats': stats,
        'mean_returns': mean_returns,
        'cov_matrix': cov_matrix,
        'rf_rate': rf_rate,
        'assets': assets,
        'target_returns': target_returns_range
    }

# --- 2. UI and Main Handler ---
comparison_all_tickers = ['AMD', 'AMZN', 'F', 'JPM', 'NVDA', 'PFE', 'TGT', 'XOM', 'WMT', 'VZ']
comparison_default_start_date = datetime.date(2002, 12, 1)
comparison_last_month_end = (pd.Timestamp.today().normalize().replace(day=1) - pd.Timedelta(days=1)).date()
comparison_start_box = widgets.Text(value=comparison_default_start_date.isoformat(), description='Start:')
comparison_through_box = widgets.Text(value=comparison_last_month_end.isoformat(), description='Through:')
comparison_checkbox_widgets = {
    ticker: widgets.Checkbox(value=True, description=ticker, indent=False,
                             layout=widgets.Layout(width='auto'))
    for ticker in comparison_all_tickers
}
comparison_run_button = widgets.Button(description='Download and run', button_style='primary')
comparison_output_tables, comparison_output_plots = widgets.Output(), widgets.Output()

def run_comparison_analysis(start, through):
    selected_tickers = [
        ticker for ticker, checkbox in comparison_checkbox_widgets.items() if checkbox.value
    ]
    if len(selected_tickers) < 2:
        print('Select at least two assets.')
        return None
    if start >= through:
        print('The start date must be earlier than the through-date.')
        return None

    end_date = through + datetime.timedelta(days=1)  # Yahoo end is exclusive.
    print(f'Downloading monthly adjusted-close data from {start} through {through}...')
    try:
        downloaded = yf.download(
            comparison_all_tickers, start=start, end=end_date, interval='1mo',
            group_by='ticker', progress=False, auto_adjust=True, actions=False
        )
        if downloaded.empty:
            raise ValueError('yf.download returned an empty DataFrame.')

        # With auto_adjust=True, Close is the adjusted-close series.
        series = []
        for ticker in comparison_all_tickers:
            if ticker in downloaded.columns.get_level_values(0):
                ticker_data = downloaded[ticker]
                if 'Close' in ticker_data.columns and ticker_data['Close'].notna().any():
                    series.append(ticker_data['Close'].rename(ticker))
        if not series:
            raise ValueError('No valid adjusted-close data was returned for any ticker.')

        adjusted_close = pd.concat(series, axis=1).sort_index().dropna(how='all')
        monthly_returns = adjusted_close.pct_change(fill_method=None).dropna(how='all')
        monthly_returns.index = pd.DatetimeIndex(monthly_returns.index).to_period('M').to_timestamp()
        min_observations = 12
        monthly_returns = monthly_returns.loc[:, monthly_returns.notna().sum() >= min_observations]
        if monthly_returns.shape[1] < 2:
            raise ValueError('At least two assets need 12 monthly returns each.')

        with warnings.catch_warnings():
            warnings.simplefilter('ignore', FutureWarning)
            ff_factors = web.DataReader(
                'F-F_Research_Data_Factors', 'famafrench', start=start, end=through
            )[0]
        rf_series = (ff_factors['RF'].astype(float) / 100).rename('rf')
        rf_series.index = ff_factors.index.to_timestamp(how='start')

        # Both scenarios use the same complete months and the same RF average.
        aligned = monthly_returns.join(rf_series, how='inner').dropna()
        if len(aligned) < min_observations:
            raise ValueError('Fewer than 12 complete months remain after aligning assets and RF.')
        available = monthly_returns.columns.tolist()
        monthly_returns = aligned[available]
        rf_rate = float(aligned['rf'].mean())
        sample_start, sample_end = monthly_returns.index.min(), monthly_returns.index.max()
        skipped = [ticker for ticker in comparison_all_tickers if ticker not in available]
        if skipped:
            print(f"Skipped unavailable or short-history tickers: {', '.join(skipped)}")
    except Exception as exc:
        print(f'Data Acquisition Error: {exc}')
        return None

    print(f'Sample: {sample_start:%Y-%m} to {sample_end:%Y-%m} '
          f'({len(monthly_returns)} months); average monthly RF: {rf_rate:.2%}')

    results_dict = {}
    for scenario_name, tickers_list in [
        ('Baseline (All Assets)', comparison_all_tickers),
        ('Current Selection', selected_tickers)
    ]:
        scenario_assets = [ticker for ticker in tickers_list if ticker in available]
        if len(scenario_assets) < 2:
            results_dict[scenario_name] = None
            continue
        try:
            results_dict[scenario_name] = analyze_comparison_period(
                monthly_returns[scenario_assets], rf_rate
            )
        except Exception as exc:
            print(f'Error in {scenario_name}: {exc}')
            results_dict[scenario_name] = None

    # --- Display Tables Side-by-Side ---
    with comparison_output_tables:
        html_tables = ""
        
        for title in ['Baseline (All Assets)', 'Current Selection']:
            r = results_dict[title]
            if r is None:
                html_tables += f"<div style='flex:1; padding:10px;'><p style='color:orange;'>Insufficient data for {title}.</p></div>"
                continue
            
            port_cols = {
                'Global MV (No Short)': r['stats']['gmvp_ns']['w'],
                'Global MV (Short)': r['stats']['gmvp_s']['w'],
                'Optimal (No Short)': r['stats']['opt_ns']['w'],
                'Optimal (Short)': r['stats']['opt_s']['w']
            }
            res_df = pd.DataFrame(port_cols, index=r['assets'])
            
            summary_data = {
                'Global MV (No Short)': [
                    r['stats']['gmvp_ns']['er'], r['stats']['gmvp_ns']['std'],
                    r['rf_rate'], r['stats']['gmvp_ns']['sharpe']
                ],
                'Global MV (Short)': [
                    r['stats']['gmvp_s']['er'], r['stats']['gmvp_s']['std'],
                    r['rf_rate'], r['stats']['gmvp_s']['sharpe']
                ],
                'Optimal (No Short)': [
                    r['stats']['opt_ns']['er'], r['stats']['opt_ns']['std'],
                    r['rf_rate'], r['stats']['opt_ns']['sharpe']
                ],
                'Optimal (Short)': [
                    r['stats']['opt_s']['er'], r['stats']['opt_s']['std'],
                    r['rf_rate'], r['stats']['opt_s']['sharpe']
                ]
            }
            summary_df = pd.DataFrame(summary_data, index=['E(rp)', 'σp', 'rf', 'Sharpe Ratio'])
            final_table = pd.concat([res_df, summary_df])
            styler = final_table.style.format('{:.2%}').format(
                subset=pd.IndexSlice[['Sharpe Ratio'], :], formatter="{:.4f}"
            ).set_caption(title)
            html_tables += f"<div style='flex:1; padding:0 10px;'>{styler.to_html()}</div>"
        
        display(HTML(f"<h3>Opportunity Set Comparison</h3><div style='display: flex;'>{html_tables}</div><hr>"))

    # --- Display Plots Side-by-Side ---
    with comparison_output_plots:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
        
        # Include the assets, frontiers, RF, and portfolio markers in the axes.
        max_x, max_y, min_y = 0, -np.inf, np.inf
        for r in results_dict.values():
            if r is not None:
                plotted_risks = np.r_[
                    np.sqrt(np.diag(r['cov_matrix'].values)),
                    r['stats']['frontier_s'], r['stats']['frontier_ns'],
                    [point['std'] for point in r['stats'].values()
                     if isinstance(point, dict) and 'std' in point]
                ]
                finite_risks = plotted_risks[np.isfinite(plotted_risks)]
                max_x = max(max_x, finite_risks.max() * 1.1)
                plotted_returns = np.r_[
                    r['mean_returns'].values, r['target_returns'], [r['rf_rate']],
                    [point['er'] for point in r['stats'].values()
                     if isinstance(point, dict) and 'er' in point]
                ]
                finite_returns = plotted_returns[np.isfinite(plotted_returns)]
                max_y = max(max_y, finite_returns.max())
                min_y = min(min_y, finite_returns.min())
        y_pad = max((max_y - min_y) * 0.05, 0.001)

        for ax, title_text in zip([ax1, ax2], ['Baseline (All Assets)', 'Current Selection']):
            r = results_dict[title_text]
            
            if r is None:
                ax.text(0.5, 0.5, 'Insufficient Data', ha='center', va='center', transform=ax.transAxes)
                ax.set_title(title_text)
                continue
            
            stats = r['stats']
            ax.plot(stats['frontier_s'], r['target_returns'], linestyle='--', c='gray', label='MVF (Shorts)')
            ax.plot(stats['frontier_ns'], r['target_returns'], linestyle='-',
                    c='lightsteelblue', linewidth=1.5, label='MVF (No Shorts)')
            efficient_ns = np.where(
                r['target_returns'] >= stats['gmvp_ns']['er'] - 1e-9,
                stats['frontier_ns'], np.nan
            )
            ax.plot(efficient_ns, r['target_returns'], c='blue', linewidth=2.5,
                    label='EF (No Shorts)')
            
            # Extract diagonal from .values
            asset_risk = np.sqrt(np.diag(r['cov_matrix'].values))
            ax.scatter(asset_risk, r['mean_returns'], marker='o', s=60, label='Assets')
            
            for ticker in r['assets']:
                ticker_std = np.sqrt(r['cov_matrix'].loc[ticker, ticker])
                ticker_return = r['mean_returns'].loc[ticker]
                ax.annotate(ticker, (ticker_std * 1.02, ticker_return))
            
            port_points = {
                'GMVP (No Short)': (stats['gmvp_ns']['std'], stats['gmvp_ns']['er']),
                'GMVP (Short)': (stats['gmvp_s']['std'], stats['gmvp_s']['er']),
                'Optimal (No Short)': (stats['opt_ns']['std'], stats['opt_ns']['er']),
                'Optimal (Short)': (stats['opt_s']['std'], stats['opt_s']['er'])
            }
            colors = ['darkorange', 'red', 'purple', 'green']
            for (name, (std, er)), color in zip(port_points.items(), colors):
                ax.scatter(std, er, marker='*', s=250, zorder=5, label=name, color=color)
            
            cml_x = np.array([0, max_x])
            ax.plot(cml_x, r['rf_rate'] + cml_x * stats['opt_ns']['sharpe'], color='purple', linestyle='--', label='CAL (No Shorts)')
            ax.plot(cml_x, r['rf_rate'] + cml_x * stats['opt_s']['sharpe'], color='green', linestyle=':', label='CAL (Shorts)')
            
            ax.set_title(f"{title_text} ({len(r['assets'])} assets; "f"{sample_start:%Y-%m} to {sample_end:%Y-%m})", fontsize=14)
            ax.set_xlabel('Monthly Standard Deviation (Risk)')
            ax.set_ylabel('Mean Monthly Total Return')
            ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.2%}'))
            ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.2%}'))
            ax.legend()
            ax.grid(True, linestyle=':', alpha=0.6)
            ax.set_xlim(left=0, right=max_x)
            ax.set_ylim(bottom=min(0, min_y - y_pad), top=max_y + y_pad)
        
        plt.tight_layout()
        plt.show()

    return results_dict

def on_comparison_run(_):
    with comparison_output_tables:
        clear_output(wait=True)
    with comparison_output_plots:
        clear_output(wait=True)
    try:
        start = datetime.date.fromisoformat(comparison_start_box.value.strip().replace('.', '-'))
        through = datetime.date.fromisoformat(comparison_through_box.value.strip().replace('.', '-'))
    except ValueError:
        with comparison_output_tables:
            print('Invalid date. Use YYYY-MM-DD or YYYY.MM.DD.')
        return
    with comparison_output_tables:
        run_comparison_analysis(start, through)

comparison_run_button.on_click(on_comparison_run)
comparison_boxes = list(comparison_checkbox_widgets.values())
comparison_midpoint = len(comparison_boxes) // 2
comparison_checkbox_grid = widgets.VBox([
    widgets.HBox(comparison_boxes[:comparison_midpoint]),
    widgets.HBox(comparison_boxes[comparison_midpoint:])
])
display(widgets.VBox([
    widgets.HBox([comparison_start_box, comparison_through_box]),
    widgets.Label('Select Assets:'),
    comparison_checkbox_grid,
    comparison_run_button,
    comparison_output_tables,
    comparison_output_plots
]))


### Capital Allocation to Risk-free and Risky Assets
* Each CAL is uniquely identified by its slope.

* The optimal risky portfolio is the tangency portfolio
  * The unique portfolio with the highest Sharpe-Ratio.

* The optimal risky portfolio does not involve the degree of risk aversion of any individual investor.

* Every investor, regardless of her/his level of risk aversion, will agree on the best CAL, and allocate her/his wealth between risk-free asset and the optimal risky portfolio.

* The portion invested in the optimal risky portfolio, however, will depend on each investor's degree of risk aversion.

---



In [8]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# --- 1. Define Asset and Market Parameters ---
E_rA, std_A = 0.08, 0.12
E_rB, std_B = 0.18, 0.30
correlation_AB = 0.15
rf_rate = 0.05

# --- 2. Core Calculation Functions ---
def portfolio_performance(w_A, w_B):
    E_rp = w_A * E_rA + w_B * E_rB
    cov_AB = correlation_AB * std_A * std_B
    var_p = w_A**2 * std_A**2 + w_B**2 * std_B**2 + 2 * w_A * w_B * cov_AB
    return E_rp, np.sqrt(var_p)

# --- 3. High-Quality Plotting Function ---
def generate_professional_plot(risk_aversion_A):
    fig, ax = plt.subplots(figsize=(10, 8))
    weights_A = np.linspace(-0.5, 1.5, 200)
    frontier_returns, frontier_stds = np.array([portfolio_performance(w, 1-w) for w in weights_A]).T
    gmvp_idx = np.argmin(frontier_stds)
    
    ax.plot(frontier_stds * 100, frontier_returns * 100, color='black', linewidth=1.5)
    ax.plot(frontier_stds[:gmvp_idx + 1] * 100, frontier_returns[:gmvp_idx + 1] * 100, color='black', linewidth=2.5)
    sharpe_ratios = (frontier_returns - rf_rate) / frontier_stds
    optimal_risky_idx = np.nanargmax(sharpe_ratios)
    E_rP, std_P = frontier_returns[optimal_risky_idx], frontier_stds[optimal_risky_idx]
    sharpe_P = (E_rP - rf_rate) / std_P
    cal_x = np.array([0, 0.30])
    cal_y = rf_rate + sharpe_P * cal_x
    ax.plot(cal_x * 100, cal_y * 100, color='deepskyblue', linewidth=2.5, zorder=3)
    y_star = (E_rP - rf_rate) / (risk_aversion_A * std_P**2)
    y_star = np.clip(y_star, 0, 2)
    E_rC = y_star * E_rP + (1 - y_star) * rf_rate
    std_C = y_star * std_P
    utility_C = E_rC - 0.5 * risk_aversion_A * std_C**2
    indiff_curve_stds = np.linspace(0.01, 0.25, 100)
    indiff_curve_returns = utility_C + 0.5 * risk_aversion_A * indiff_curve_stds**2
    ax.plot(indiff_curve_stds * 100, indiff_curve_returns * 100, color='black', linewidth=1.5)
    ax.scatter([std_P*100, std_C*100], [E_rP*100, E_rC*100], color='black', s=100, zorder=5)
    ax.scatter([0.20*100], [(rf_rate + sharpe_P * 0.20)*100], color='black', s=100, zorder=5)
    ax.scatter([std_A*100], [8.7], color='black', s=100, zorder=5)
    
    ax.annotate('P (Optimal Risky Portfolio)', (std_P*100, E_rP*100), xytext=(25, 10.5), textcoords='offset points', arrowprops=dict(arrowstyle="->"))
    ax.annotate('C (Optimal Complete Portfolio)', (std_C*100, E_rC*100), xytext=(-150, 0), textcoords='offset points', arrowprops=dict(arrowstyle="->"))
    ax.text(std_A*100 + 0.5, 8.7, 'D')
    ax.text(0.20*100 + 0.5, (rf_rate + sharpe_P * 0.20)*100, 'E')
    ax.plot([std_C*100, std_C*100], [0, E_rC*100], 'k--')
    ax.plot([0, std_C*100], [E_rC*100, E_rC*100], 'k--')
    ax.plot([std_P*100, std_P*100], [0, E_rP*100], 'k--')
    
    ax.set_xlabel('Standard Deviation (%)', fontsize=12)
    ax.set_ylabel('Expected Return (%)', fontsize=12)
    ax.set_xlim(0, 30)
    ax.set_ylim(0, 18)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.text(-0.5, 5, r'$r_f = 5\%$', fontsize=12, horizontalalignment='right')
    
    plt.show()

# --- 4. Interactive Widgets Setup ---
risk_aversion_slider = widgets.FloatSlider(
    value=4.0, min=2.0, max=8.0, step=0.25,
    description='Risk Aversion (A):', continuous_update=True,
    style={'description_width': 'initial'}, layout=widgets.Layout(width='50%')
)
interactive_plot = widgets.interactive_output(generate_professional_plot, {'risk_aversion_A': risk_aversion_slider})

# --- Add Header and Display ---
header = widgets.HTML("<h3>Capital Allocation to Risk-free and Risky Assets</h3>")
ui = widgets.VBox([header, risk_aversion_slider, interactive_plot])
display(ui)

In [9]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# --- 1. Define Fixed Parameters for the Risky Portfolio P ---
E_rP = 0.15
std_P = 0.22

# --- 2. Plotting Function ---
def generate_cal_plots(lending_rate_pct, borrowing_rate_pct):
    lending_rate = lending_rate_pct / 100
    borrowing_rate = borrowing_rate_pct / 100

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    # --- Plot 1: Standard CAL (Single Rate) ---
    sharpe_lending = (E_rP - lending_rate) / std_P
    cal1_x = np.array([0, std_P * 1.5])
    cal1_y = lending_rate + cal1_x * sharpe_lending
    
    ax1.plot(cal1_x, cal1_y, color='deepskyblue', linewidth=2)
    ax1.scatter([0, std_P], [lending_rate, E_rP], color='black', s=80, zorder=5)
    
    # Annotations for Plot 1
    ax1.text(0, lending_rate, ' F', fontsize=12, verticalalignment='top')
    ax1.text(std_P, E_rP, ' P', fontsize=12, verticalalignment='bottom')
    ax1.text(std_P * 1.1, cal1_y[-1], 'CAL = Capital\nAllocation\nLine', fontsize=11, verticalalignment='center')
    # --- FIX: Added the calculated result to the annotation ---
    ax1.text(std_P * 0.5, lending_rate + 0.01, f'S = {E_rP - lending_rate:.2f}/{std_P:.2f} = {sharpe_lending:.2f}', fontsize=11)
    
    # Dashed lines for Plot 1
    ax1.plot([0, std_P], [E_rP, E_rP], 'k--', alpha=0.7)
    ax1.plot([std_P, std_P], [0, E_rP], 'k--', alpha=0.7)
    
    ax1.set_xlabel('σ', fontsize=12)
    ax1.set_ylabel('E(r)', fontsize=12, rotation=0, labelpad=15)
    
    # --- Plot 2: Kinked CAL (Different Borrowing/Lending Rates) ---
    sharpe_borrowing = (E_rP - borrowing_rate) / std_P
    
    cal2_lend_x = np.array([0, std_P])
    cal2_lend_y = lending_rate + cal2_lend_x * sharpe_lending
    ax2.plot(cal2_lend_x, cal2_lend_y, color='deepskyblue', linewidth=2)
    
    cal2_borrow_x = np.array([std_P, std_P * 1.5])
    cal2_borrow_y = E_rP + (cal2_borrow_x - std_P) * sharpe_borrowing
    ax2.plot(cal2_borrow_x, cal2_borrow_y, color='deepskyblue', linewidth=2)
    
    ax2.scatter([0, 0, std_P], [lending_rate, borrowing_rate, E_rP], color='black', s=80, zorder=5)
    
    ax2.text(std_P, E_rP, ' P', fontsize=12, verticalalignment='bottom')
    ax2.text(std_P * 0.5, lending_rate + 0.01, f'S(y ≤ 1) = {sharpe_lending:.2f}', fontsize=11)
    ax2.text(std_P * 1.1, E_rP + 0.01, f'S(y > 1) = {sharpe_borrowing:.2f}', fontsize=11)
    
    ax2.plot([0, std_P], [E_rP, E_rP], 'k--', alpha=0.7)
    ax2.plot([std_P, std_P], [0, E_rP], 'k--', alpha=0.7)
    ax2.plot([0, std_P * 1.5], [borrowing_rate, borrowing_rate + (std_P*1.5)*sharpe_borrowing], 'k--', alpha=0.5)

    ax2.set_xlabel('σ', fontsize=12)
    ax2.set_ylabel('E(r)', fontsize=12, rotation=0, labelpad=15)

    for ax in [ax1, ax2]:
        ax.set_xlim(0, 0.30)
        ax.set_ylim(0, 0.20)
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    fig.text(0.5, -0.05, f'Lending range slope = {(E_rP - lending_rate):.2f}/{std_P:.2f} = {sharpe_lending:.2f}\n'
                         f'Borrowing range slope = {(E_rP - borrowing_rate):.2f}/{std_P:.2f} = {sharpe_borrowing:.2f}',
             ha='center', fontsize=12)
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

# --- 3. Interactive Widgets Setup ---
style = {'description_width': 'initial'}
# --- FIX: Set default values to 8.0 ---
lending_rate_slider = widgets.FloatSlider(value=8.0, min=1, max=10, step=0.5, description='Lending Rate (%):', style=style, continuous_update=True)
borrowing_rate_slider = widgets.FloatSlider(value=8.0, min=8.0, max=15, step=0.5, description='Borrowing Rate (%):', style=style, continuous_update=True)

def keep_borrowing_rate_valid(change):
    borrowing_rate_slider.min = change['new']

lending_rate_slider.observe(keep_borrowing_rate_valid, names='value')

interactive_plot = widgets.interactive_output(generate_cal_plots, {
    'lending_rate_pct': lending_rate_slider,
    'borrowing_rate_pct': borrowing_rate_slider
})

header = widgets.HTML("<h3>The Opportunity Set with Different Borrowing and Lending Rates</h3>")
ui = widgets.VBox([header, widgets.HBox([lending_rate_slider, borrowing_rate_slider]), interactive_plot])
display(ui)

### Finding the Optimal Risky Portfolio

Consider two risky funds:

| Fund | Expected return | Standard deviation |
|:--|--:|--:|
| Debt (bond) fund | 8% | 12% |
| Equity fund | 13% | 20% |

Their return correlation is 0.30, and the risk-free rate is 5%.

* The first interactive shows how changing the equity weight from 0% to 100% changes the risky portfolio and its CAL.

* The second calculates the fund weights that maximize the Sharpe ratio, $S_p=[E(r_p)-r_f]/\sigma_p$, and displays the variance breakdown.

---


In [10]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown, Math

# --- 1. Define Asset Parameters ---
E_rD, std_D = 0.08, 0.12  # Debt Fund
E_rE, std_E = 0.13, 0.20  # Equity Fund
correlation_DE = 0.30

# --- 2. Core Calculation Functions ---
def investment_portfolio_performance(w_E):
    w_D = 1 - w_E
    E_rp = w_D * E_rD + w_E * E_rE
    cov_DE = correlation_DE * std_D * std_E
    var_p = w_D**2 * std_D**2 + w_E**2 * std_E**2 + 2 * w_D * w_E * cov_DE
    return E_rp, np.sqrt(var_p)

# --- 3. Combined Plotting and Info Function ---
def generate_output(weight_in_equity_pct, rf_rate_pct):
    weight_in_equity = weight_in_equity_pct / 100
    rf_rate = rf_rate_pct / 100
    
    # --- Calculate portfolio values ---
    E_r_selected, std_selected = investment_portfolio_performance(weight_in_equity)
    
    # Find the Optimal Risky Portfolio (Tangency Portfolio)
    weights_E_frontier = np.linspace(0, 1, 100)
    frontier_returns, frontier_stds = np.array([investment_portfolio_performance(w) for w in weights_E_frontier]).T
    cov_DE = correlation_DE * std_D * std_E
    premium_D, premium_E = E_rD - rf_rate, E_rE - rf_rate
    denominator = (premium_E * std_D**2 + premium_D * std_E**2
                   - (premium_D + premium_E) * cov_DE)
    candidate_weights = [0.0, 1.0]
    if abs(denominator) > 1e-12:
        interior_weight = (premium_E * std_D**2 - premium_D * cov_DE) / denominator
        if 0 < interior_weight < 1:
            candidate_weights.append(interior_weight)
    optimal_weight_E = max(
        candidate_weights,
        key=lambda weight: ((investment_portfolio_performance(weight)[0] - rf_rate)
                            / investment_portfolio_performance(weight)[1])
    )
    E_r_optimal, std_optimal = investment_portfolio_performance(optimal_weight_E)
    sharpe_optimal = (E_r_optimal - rf_rate) / std_optimal

    
    # --- a. Info Panel (Markdown + MathJax) ---
    info_md = f"""
#### Asset Profiles:
- Debt Fund: E(r) = {E_rD:.2%}, σ = {std_D:.2%}
- Equity Fund: E(r) = {E_rE:.2%}, σ = {std_E:.2%}

#### Selected Portfolio ({weight_in_equity_pct:.0f}% Stocks):
- E(rₚ) = {E_r_selected:.2%}
- σₚ = {std_selected:.2%}

#### Optimal Risky Portfolio (P):
- E(rₚ) = {E_r_optimal:.2%}
- σₚ = {std_optimal:.2%}
- Sharpe Ratio = {sharpe_optimal:.2f}

#### Objective:
Maximize the slope of the CAL (Sharpe Ratio):
"""
    with investment_output_info:
        clear_output(wait=True)
        display(Markdown(info_md))
        display(Math(r"S_P = \frac{E(r_p) - r_f}{\sigma_p}"))

    # --- b. Plot Generation ---
    with investment_output_plot:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(7.5, 7.5))
        ax.plot(frontier_stds * 100, frontier_returns * 100, color='deepskyblue', linewidth=2.5)

        sharpe_selected = (E_r_selected - rf_rate) / std_selected if std_selected > 0 else 0
        cal_selected_x = np.array([0, 0.25]); cal_selected_y = rf_rate + sharpe_selected * cal_selected_x
        ax.plot(cal_selected_x * 100, cal_selected_y * 100, color='black', linewidth=1.5)
        ax.scatter(std_selected * 100, E_r_selected * 100, facecolors='none', edgecolors='black', s=100, zorder=5, linewidth=1.5)

        cal_optimal_x = np.array([0, 0.25]); cal_optimal_y = rf_rate + sharpe_optimal * cal_optimal_x
        ax.plot(cal_optimal_x * 100, cal_optimal_y * 100, color='black', linewidth=1.5)
        ax.scatter(std_optimal * 100, E_r_optimal * 100, color='black', s=100, zorder=5)

        points = {'A': investment_portfolio_performance(0.18), 'B': investment_portfolio_performance(0.30), 'E': investment_portfolio_performance(0.82)}
        for label, (ret, std) in points.items():
            ax.scatter(std * 100, ret * 100, facecolors='none', edgecolors='black', s=100, zorder=5, linewidth=1.5)
            ax.text(std * 100 + 0.3, ret * 100, label, fontsize=11, verticalalignment='center')

        ax.text(investment_portfolio_performance(0.18)[1]*100, 8.0, ' D', verticalalignment='center')
        ax.plot([0, 25], [8, 8], 'k--', alpha=0.7); ax.plot([0, 25], [13, 13], 'k--', alpha=0.7)
        ax.set_title('The Investment Decision', fontsize=14)
        ax.set_xlabel('Standard Deviation (%)', fontsize=12); ax.set_ylabel('Expected Return (%)', fontsize=12)
        ax.set_xlim(0, 25); ax.set_ylim(5, 14)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
        ax.tick_params(axis='both', which='major', labelsize=11)
        plt.show()

# --- 4. Interactive Widgets Setup ---
style = {'description_width': 'initial'}
equity_weight_slider = widgets.FloatSlider(value=18.0, min=0, max=100, step=1, description='Weight in Stocks (%):', style=style, continuous_update=True)
rf_rate_slider = widgets.FloatSlider(value=5.0, min=2, max=8, step=0.25, description='Risk-Free Rate (%):', style=style, continuous_update=True)
investment_output_info = widgets.Output(layout=widgets.Layout(overflow='auto'))
investment_output_plot = widgets.Output()
io_link = widgets.interactive_output(generate_output, {'weight_in_equity_pct': equity_weight_slider, 'rf_rate_pct': rf_rate_slider})

# --- 5. Side-by-side Layout ---
header = widgets.HTML("<h3>Finding the Optimal Risky Portfolio</h3>")
controls = widgets.VBox([equity_weight_slider, rf_rate_slider], layout=widgets.Layout(gap='10px'))
left_col = widgets.VBox([header, controls, investment_output_info], layout=widgets.Layout(width='45%', padding='0 12px 0 0'))
right_col = widgets.VBox([investment_output_plot], layout=widgets.Layout(width='55%'))
ui = widgets.HBox([left_col, right_col], layout=widgets.Layout(align_items='flex-start'))

display(ui)
generate_output(equity_weight_slider.value, rf_rate_slider.value)

In [11]:
import numpy as np
import pandas as pd
import ipywidgets as widgets
from scipy.optimize import minimize
from IPython.display import display, clear_output, HTML, Markdown

# --- 1. UI Widgets (FloatText for precise input) ---
style = {'description_width': 'initial'}
layout = widgets.Layout(width='200px')
# Asset Inputs
er_d_widget = widgets.BoundedFloatText(value=8.00, min=0, max=100, description='E(r) Bonds (%):', style=style, step=0.1, layout=layout)
std_d_widget = widgets.BoundedFloatText(value=12.00, min=0, max=100, description='σ Bonds (%):', style=style, step=0.1, layout=layout)
er_e_widget = widgets.BoundedFloatText(value=13.00, min=0, max=100, description='E(r) Equity (%):', style=style, step=0.1, layout=layout)
std_e_widget = widgets.BoundedFloatText(value=20.00, min=0, max=100, description='σ Equity (%):', style=style, step=0.1, layout=layout)
# Market/Portfolio Inputs
corr_widget = widgets.BoundedFloatText(value=0.30, min=-1, max=1, description='Correlation (ρ):', style=style, step=0.1, layout=layout)
rf_widget = widgets.BoundedFloatText(value=5.00, min=0, max=100, description='Risk-Free Rate (%):', style=style, step=0.1, layout=layout)
# Optimization Button
optimize_button = widgets.Button(description="Optimize Portfolio", button_style='success', icon='calculator')
# Output area
output_area = widgets.Output()

# --- 2. Optimization and Display Function ---
def optimize_and_display(b=None):
    with output_area:
        clear_output(wait=True)
        
        # --- a. Get inputs and calculate base metrics ---
        try:
            er_d, std_d, er_e, std_e, rf_rate = [w.value/100 for w in [er_d_widget, std_d_widget, er_e_widget, std_e_widget, rf_widget]]
            corr = corr_widget.value
            mean_returns = np.array([er_d, er_e])
            cov_matrix = np.array([[std_d**2, corr*std_d*std_e], [corr*std_d*std_e, std_e**2]])
            num_assets = 2
            initial_guess = num_assets * [1./num_assets]
        except (TypeError, ZeroDivisionError) as e:
            display(HTML(f"<p style='color:red;'>Input Error: Please ensure all fields have valid numbers. Details: {e}</p>"))
            return
            
        # --- b. Define objective function and constraints ---
        def neg_sharpe_ratio(weights, mean_returns, cov_matrix, rf_rate):
            er_p = weights.T @ mean_returns
            std_p = np.sqrt(weights.T @ cov_matrix @ weights)
            return -(er_p - rf_rate) / std_p if std_p > 0 else np.inf

        constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
        bounds = tuple((0, 1) for _ in range(num_assets))

        # --- c. Run optimizer ---
        result = minimize(neg_sharpe_ratio, initial_guess, args=(mean_returns, cov_matrix, rf_rate),
                          method='SLSQP', bounds=bounds, constraints=constraints, options={'ftol': 1e-12})
        
        if not result.success:
            display(HTML(f"<p style='color:orange;'>Optimization did not converge: {result.message}</p>"))
            return
            
        candidates = [result.x, np.array([1.0, 0.0]), np.array([0.0, 1.0])]
        w_d, w_e = max(
            candidates,
            key=lambda w: -neg_sharpe_ratio(w, mean_returns, cov_matrix, rf_rate)
        )
        
        # --- d. Calculate final portfolio metrics ---
        er_p = w_d * er_d + w_e * er_e
        var_p = w_d**2 * std_d**2 + w_e**2 * std_e**2 + 2 * w_d * w_e * (corr * std_d * std_e)
        std_p = np.sqrt(var_p)
        sharpe_p = (er_p - rf_rate) / std_p

        # --- e. Create and style DataFrames for display ---
        summary_df = pd.DataFrame({
            'Metric': ['Portfolio Mean', 'Portfolio Variance', 'Portfolio Std Dev', 'Risk-Free Rate', 'Sharpe Ratio'],
            'Value': [f"{er_p:.2%}", f"{var_p:.6f}", f"{std_p:.2%}", f"{rf_rate:.2%}", f"{sharpe_p:.4f}"]
        })
        
        cov_matrix_df = pd.DataFrame(cov_matrix, index=['Bonds', 'Equity'], columns=['Bonds', 'Equity'])
        
        contrib_data = {'Bonds': [w_d**2 * std_d**2, w_e*w_d*(corr*std_d*std_e)], 'Equity': [w_d*w_e*(corr*std_d*std_e), w_e**2 * std_e**2]}
        contrib_df = pd.DataFrame(contrib_data, index=[f"{w_d:.2%} Bonds", f"{w_e:.2%} Equity"])
        contrib_df.loc['Total'] = contrib_df.sum()
        contrib_df['Total'] = contrib_df.sum(axis=1)

        # --- f. Convert to HTML and display ---
        styler_sum = summary_df.style.hide(axis='index').set_table_attributes('style="width:100%; margin-bottom: 20px;"').set_properties(**{'text-align': 'left'}).set_table_styles([
            {'selector': 'th', 'props': [('text-align', 'center')]},
            {'selector': 'td', 'props': [('text-align', 'center')]}
        ])
        styler_cov = cov_matrix_df.style.format('{:.6f}').set_caption("Covariance Matrix").set_table_attributes('style="width:100%;"').set_table_styles([
            {'selector': 'th', 'props': [('text-align', 'center')]},
            {'selector': 'td', 'props': [('text-align', 'center')]}
        ])
        styler_contrib = contrib_df.style.format('{:.6f}').set_caption("Portfolio Variance Contributions").set_table_attributes('style="width:100%;"').set_table_styles([
            {'selector': 'th', 'props': [('text-align', 'center')]},
            {'selector': 'td', 'props': [('text-align', 'center')]}
        ])
        
        formulas_md = r"""
### Portfolio Formulas
$$E(r_p) = \sum w_i E(r_i)$$

$$\sigma^2(r_p) = w_1^2 \sigma^2(r_1) + w_2^2 \sigma^2(r_2) + 2 w_1 w_2 \text{Cov}(r_1, r_2)$$

$$\rho_{i,j} = \frac{\text{Cov}(r_i, r_j)}{\sigma_i \sigma_j}$$
"""
        
        display(HTML(f"<h2 style='text-align: center; color: #2c5282; margin: 20px 0;'>Optimal Weights: {w_d:.2%} Bonds | {w_e:.2%} Equity</h2>"))
        
        display(HTML(f"""
            <div style='display: grid; grid-template-columns: 1fr 1fr; gap: 30px; margin: 20px 0;'>
                <div>
                    <h3 style='color: #2d3748; margin-bottom: 10px;'>Portfolio Metrics</h3>
                    {styler_sum.to_html()}
                </div>
                <div>
                    <h3 style='color: #2d3748; margin-bottom: 10px;'>Covariance Matrix</h3>
                    {styler_cov.to_html()}
                </div>
            </div>
        """))
        
        display(HTML(f"<div style='margin: 30px 0;'><h3 style='color: #2d3748; margin-bottom: 10px;'>Variance Breakdown</h3>{styler_contrib.to_html()}</div>"))
        display(Markdown(formulas_md))

# --- 3. Assemble and Link UI ---
optimize_button.on_click(optimize_and_display)
asset_controls = widgets.VBox([er_d_widget, std_d_widget, er_e_widget, std_e_widget])
portfolio_controls = widgets.VBox([corr_widget, rf_widget])
controls = widgets.VBox([widgets.HBox([asset_controls, portfolio_controls]), optimize_button], layout=widgets.Layout(align_items='center'))

display(widgets.VBox([controls, output_area]))
optimize_and_display()

### The Investment Decision: Completing the Portfolio

From the preceding example, the optimal risky portfolio $P$ is 40% debt fund and 60% equity fund. 

* Thus $E(r_P)=11\%$, $\sigma_P^2=0.02016$, and $\sigma_P\approx14.20\%$.

* If $y$ is the weight in $P$ and the investor can lend or borrow at $r_f=5\%$,

$$E(r_C)=r_f+y[E(r_P)-r_f], \qquad \sigma_C=y\sigma_P.$$

* For risk-aversion coefficient $A$, maximize

$$U(y)=r_f+y[E(r_P)-r_f]-\frac{1}{2}Ay^2\sigma_P^2
\quad\Rightarrow\quad
 y^*=\frac{E(r_P)-r_f}{A\sigma_P^2}.$$

* At $A=4$,

$$y^*=\frac{0.11-0.05}{4(0.02016)}=0.7440.$$

* Invest 74.40% in $P$ and 25.60% in the risk-free asset. Then $E(r_C)\approx9.46\%$ and $\sigma_C\approx10.56\%$.

---



In [12]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown, Math

# --- 1. Define Underlying Asset and Market Parameters ---
complete_E_rD, complete_std_D = 0.08, 0.12  # Debt Fund
complete_E_rE, complete_std_E = 0.13, 0.20  # Equity Fund
complete_correlation_DE = 0.30
complete_rf_rate = 0.05 # Risk-free rate = 5%

# --- 2. Core Calculation Functions ---
def complete_portfolio_performance(w_E):
    """Calculates return and std dev for a given weight in the Equity fund."""
    w_D = 1 - w_E
    E_rp = w_D * complete_E_rD + w_E * complete_E_rE
    cov_DE = complete_correlation_DE * complete_std_D * complete_std_E
    var_p = w_D**2 * complete_std_D**2 + w_E**2 * complete_std_E**2 + 2 * w_D * w_E * cov_DE
    return E_rp, np.sqrt(var_p)

# --- 3. Combined Plotting and Info Function ---
def generate_complete_output(risk_aversion_A):
    # --- a. Calculate the risky opportunity set and optimal risky portfolio (P) ---
    weights_E_frontier = np.linspace(0, 1, 501) # Weights for Equity fund
    frontier_returns, frontier_stds = np.array([complete_portfolio_performance(w) for w in weights_E_frontier]).T
    
    # Find the Optimal Risky Portfolio (P) by maximizing the Sharpe Ratio
    sharpe_ratios = (frontier_returns - complete_rf_rate) / frontier_stds
    optimal_idx = np.nanargmax(sharpe_ratios)
    E_rP, std_P = frontier_returns[optimal_idx], frontier_stds[optimal_idx]
    var_P = std_P**2

    # --- b. Calculate the Optimal Allocation (y*) and Complete Portfolio (C) ---
    y_star = (E_rP - complete_rf_rate) / (risk_aversion_A * var_P)
    y_star_clipped = np.clip(y_star, 0, 2)
    E_rC = y_star_clipped * E_rP + (1 - y_star_clipped) * complete_rf_rate
    std_C = y_star_clipped * std_P

    # --- c. Info Panel ---
    info_md = f"""
#### Optimal Risky Portfolio (P):
- $E(r_P)$ = {E_rP:.2%}
- $\\sigma_P$ = {std_P:.2%}; $\\sigma_P^2$ = {var_P:.5f}

#### Optimal Allocation to Complete Portfolio (C):
The optimal weight, $y^*$, in the risky portfolio P is found by maximizing the utility function:
"""
    y_star_formula = r"y^* = \frac{E(r_P) - r_f}{A \sigma_P^2}"
    y_star_calc = f"""
For A = {risk_aversion_A:.2f}:
$$y^* = \\frac{{{E_rP-complete_rf_rate:.4f}}}{{{risk_aversion_A:.2f} \\times {var_P:.5f}}} = {y_star:.4f}$$
Resulting Complete Portfolio:
- $E(r_C)$ = {E_rC:.2%}
- $\\sigma_C$ = {std_C:.2%}
"""
    with complete_output_info:
        clear_output(wait=True)
        display(Markdown(info_md))
        display(Math(y_star_formula))
        display(Markdown(y_star_calc))

    # --- d. Plot Generation ---
    with complete_output_plot:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(7.5, 7.5))
        
        # Plot only the efficient branch, starting at the global minimum-variance portfolio
        gmvp_idx = np.argmin(frontier_stds)
        ax.plot(frontier_stds[gmvp_idx:] * 100, frontier_returns[gmvp_idx:] * 100, color='deepskyblue', linewidth=2.5, label='Efficient Frontier')
        
        # Plot the Capital Allocation Line (CAL)
        sharpe_P = (E_rP - complete_rf_rate) / std_P
        cal_x = np.array([0, 0.25]); cal_y = complete_rf_rate + sharpe_P * cal_x
        ax.plot(cal_x * 100, cal_y * 100, color='black', linewidth=1.5, label='Capital Allocation Line')
        
        # Plot the Indifference Curve
        utility_C = E_rC - 0.5 * risk_aversion_A * std_C**2
        indiff_curve_stds = np.linspace(0.01, 0.25, 100)
        indiff_curve_returns = utility_C + 0.5 * risk_aversion_A * indiff_curve_stds**2
        ax.plot(indiff_curve_stds * 100, indiff_curve_returns * 100, color='black', linewidth=1.5, label='Indifference Curve')

        # Plot points P and C
        ax.scatter([std_P*100, std_C*100], [E_rP*100, E_rC*100], color='black', s=100, zorder=5)
        ax.annotate('P', (std_P*100 + 0.3, E_rP*100))
        ax.annotate('C', (std_C*100 + 0.3, E_rC*100))
        
        ax.set_title('The Investment Decision', fontsize=14)
        ax.set_xlabel('Standard Deviation (%)'); ax.set_ylabel('Expected Return (%)')
        ax.set_xlim(0, 25); ax.set_ylim(0, 18)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
        plt.show()

# --- 4. Interactive Widgets Setup ---
risk_aversion_slider = widgets.FloatSlider(value=4.0, min=2.0, max=8.0, step=0.25, description='Risk Aversion (A):', continuous_update=True, style={'description_width': 'initial'})
complete_output_info = widgets.Output(layout=widgets.Layout(overflow='auto'))
complete_output_plot = widgets.Output()
io_link = widgets.interactive_output(generate_complete_output, {'risk_aversion_A': risk_aversion_slider})

# --- 5. Side-by-side Layout ---
header = widgets.HTML("<h3>Finding the Optimal Complete Portfolio</h3>")
controls = widgets.VBox([risk_aversion_slider])
left_col = widgets.VBox([header, controls, complete_output_info], layout=widgets.Layout(width='45%', padding='0 12px 0 0'))
right_col = widgets.VBox([complete_output_plot], layout=widgets.Layout(width='55%'))
ui = widgets.HBox([left_col, right_col], layout=widgets.Layout(align_items='flex-start'))

display(ui)
generate_complete_output(risk_aversion_slider.value)

---

### Markowitz Portfolio Optimization Model
* Security selection
  
  * Determine the risk-return opportunities available
  
  * All portfolios that lie on the minimum-variance frontier from the global minimum-variance portfolio and upward provide the best risk-return combinations
* Search for the CAL with the highest reward-to-variability ratio
  
  * Everyone invests in P, regardless of their degree of risk aversion
    
    * More risk-averse investors put less in P
    
    * Less risk-averse investors put more in P

* Capital Allocation and the Separation Property
  
  * The portfolio choice problem may be separated into two independent tasks
    
    * Determination of the optimal risky portfolio is purely technical
    
    * Allocation of the complete portfolio to risk-free versus the risky portfolio depends on personal preference

* Optimal Portfolios and Nonnormal Returns
  
  * Fat-tailed distributions can result in extreme values of VaR and ES.
  
  * If other portfolios provide sufficiently better VaR and ES values than the mean-variance efficient portfolio, we may prefer these when faced with fat-tailed distributions.

---



### Markowitz Portfolio Optimization Model
* Passive Strategies: The Capital Market Line
  
  * Determination of the assets to include in P may result from a passive or an active strategy.
  
  * A passive strategy describes a portfolio decision that avoids any direct or indirect security analysis.
  
  * A natural candidate for a passively held risky asset would be a well-diversified portfolio of common stocks.

| Period | U.S. Equity Market | 1-Month T-Bills | Excess Return | Standard Deviation | Sharpe Ratio |
|---|---|---|---|---|---|
| 1927-2018 | 11.72 | 3.38 | 8.34 | 20.36 | 0.41 |
| 1927-1949 | 9.40 | 0.92 | 8.49 | 26.83 | 0.32 |
| 1950-1972 | 14.00 | 3.14 | 10.86 | 17.46 | 0.62 |
| 1973-1995 | 13.38 | 7.26 | 6.11 | 18.43 | 0.33 |
| 1996-2018 | 10.10 | 2.21 | 7.89 | 18.39 | 0.43 |

---



In [13]:
# If needed, uncomment and run once:
# %pip install -U yfinance curl_cffi pandas pandas-datareader numpy scipy ipywidgets

import datetime
import warnings
import yfinance as yf
import pandas as pd
import numpy as np
import pandas_datareader.data as web
from scipy.optimize import minimize
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# --- 1. Inputs ---
mw_tickers = ['AMD', 'AMZN', 'F', 'JPM', 'NVDA', 'PFE', 'TGT', 'XOM', 'WMT', 'VZ']
mw_default_start = datetime.date(2002, 12, 1)
mw_last_month_end = (pd.Timestamp.today().normalize().replace(day=1) - pd.Timedelta(days=1)).date()
mw_start_box = widgets.Text(value=mw_default_start.isoformat(), description='Start:')
mw_through_box = widgets.Text(value=mw_last_month_end.isoformat(), description='Through:')
mw_checkboxes = {
    ticker: widgets.Checkbox(value=True, description=ticker, indent=False,
                             layout=widgets.Layout(width='auto'))
    for ticker in mw_tickers
}
mw_allow_short = widgets.Checkbox(value=False, description='Allow Short Selling')
mw_run_button = widgets.Button(description='Download and run', button_style='primary')
mw_output = widgets.Output()

# --- 2. Download, optimize, and display ---
def run_markowitz_optimizer(start_date, through_date, selected_tickers, allow_short):
    if len(selected_tickers) < 2:
        print('Select at least two assets.')
        return None
    if start_date >= through_date:
        print('The start date must be earlier than the through-date.')
        return None

    end_date = through_date + datetime.timedelta(days=1)  # Yahoo end is exclusive.
    print(f'Downloading monthly adjusted-close data from {start_date} through {through_date}...')
    try:
        downloaded = yf.download(
            selected_tickers, start=start_date, end=end_date, interval='1mo',
            group_by='ticker', progress=False, auto_adjust=True, actions=False
        )
        if downloaded.empty:
            raise ValueError('yf.download returned an empty DataFrame.')

        # With auto_adjust=True, Close is the adjusted-close series.
        series = []
        for ticker in selected_tickers:
            if ticker in downloaded.columns.get_level_values(0):
                ticker_data = downloaded[ticker]
                if 'Close' in ticker_data.columns and ticker_data['Close'].notna().any():
                    series.append(ticker_data['Close'].rename(ticker))
        if not series:
            raise ValueError('No valid adjusted-close data was returned.')

        adjusted_close = pd.concat(series, axis=1).sort_index().dropna(how='all')
        monthly_returns = adjusted_close.pct_change(fill_method=None).dropna(how='all')
        monthly_returns.index = pd.DatetimeIndex(monthly_returns.index).to_period('M').to_timestamp()
        min_observations = 12
        monthly_returns = monthly_returns.loc[:, monthly_returns.notna().sum() >= min_observations]
        if monthly_returns.shape[1] < 2:
            raise ValueError('At least two assets need 12 monthly returns each.')

        assets = monthly_returns.columns.tolist()
        skipped = [ticker for ticker in selected_tickers if ticker not in assets]
        if skipped:
            print(f"Skipped unavailable or short-history tickers: {', '.join(skipped)}")

        with warnings.catch_warnings():
            warnings.simplefilter('ignore', FutureWarning)
            ff_factors = web.DataReader(
                'F-F_Research_Data_Factors', 'famafrench',
                start=start_date, end=through_date
            )[0]
        rf_series = (ff_factors['RF'].astype(float) / 100).rename('rf')
        rf_series.index = ff_factors.index.to_timestamp(how='start')

        # Estimate asset returns and RF from the same complete months.
        aligned = monthly_returns.join(rf_series, how='inner').dropna()
        if len(aligned) < min_observations:
            raise ValueError('Fewer than 12 complete months remain after aligning assets and RF.')
        asset_returns = aligned[assets]
        rf_rate = float(aligned['rf'].mean())
    except Exception as exc:
        print(f'Data Acquisition Error: {exc}')
        return None

    sample_start, sample_end = asset_returns.index.min(), asset_returns.index.max()
    print(f'Sample: {sample_start:%Y-%m} to {sample_end:%Y-%m} '
          f'({len(asset_returns)} months); average monthly RF: {rf_rate:.2%}')

    mean_returns = asset_returns.mean()
    cov_matrix = asset_returns.cov()
    mu, Sigma = mean_returns.to_numpy(), cov_matrix.to_numpy()
    num_assets = len(assets)

    def neg_sharpe_ratio(weights):
        expected_return = float(weights @ mu)
        variance = float(weights @ Sigma @ weights)
        if variance <= 1e-12:
            return 1e12
        return -(expected_return - rf_rate) / np.sqrt(variance)

    constraints = ({'type': 'eq', 'fun': lambda weights: np.sum(weights) - 1})
    bounds = None if allow_short else [(0, 1)] * num_assets
    initial_guess = np.full(num_assets, 1 / num_assets)
    result = minimize(
        neg_sharpe_ratio, initial_guess, method='SLSQP',
        bounds=bounds, constraints=constraints,
        options={'ftol': 1e-12, 'maxiter': 1000}
    )
    if not result.success:
        print(f'Portfolio Optimization Error: {result.message}')
        return None

    optimal_weights = result.x
    er_p = float(optimal_weights @ mu)
    var_p = float(optimal_weights @ Sigma @ optimal_weights)
    if var_p <= 1e-12:
        print('Portfolio Optimization Error: portfolio variance is too small.')
        return None
    std_p = np.sqrt(var_p)
    sharpe_p = (er_p - rf_rate) / std_p

    port_stats = pd.DataFrame({
        'Monthly': [er_p, var_p, std_p, rf_rate, sharpe_p],
        'Annualized': [er_p * 12, var_p * 12, std_p * np.sqrt(12),
                       rf_rate * 12, sharpe_p * np.sqrt(12)]
    }, index=['Port Mean', 'Port Variance', 'Port Std Dev', 'RF', 'Sharpe Ratio'])
    asset_stats = pd.DataFrame({
        'Mean': mean_returns, 'StdDev': asset_returns.std(),
        'Weights': optimal_weights
    }, index=assets).T
    var_contrib = pd.DataFrame(
        np.outer(optimal_weights, optimal_weights) * Sigma,
        index=assets, columns=assets
    )

    port_styler = port_stats.style.format(
        {'Monthly': '{:.2%}', 'Annualized': '{:.2%}'},
        subset=pd.IndexSlice[['Port Mean', 'Port Std Dev', 'RF'], :]
    ).format(
        {'Monthly': '{:.4f}', 'Annualized': '{:.4f}'},
        subset=pd.IndexSlice[['Port Variance'], :]
    ).format(
        {'Monthly': '{:.4f}', 'Annualized': '{:.2f}'},
        subset=pd.IndexSlice[['Sharpe Ratio'], :]
    )
    asset_styler = asset_stats.style.format('{:.2%}')
    var_styler = var_contrib.style.format('{:.6f}').set_caption(
        'Pairwise Contributions to Monthly Portfolio Variance'
    )
    cov_styler = cov_matrix.style.format('{:.6f}').set_caption('Monthly Covariance Matrix')

    display(HTML('<h3>Markowitz Portfolio Optimization Model</h3>'))
    print(f"Assets used: {', '.join(assets)}")
    display(HTML('<h4>Portfolio Statistics</h4>'), port_styler)
    display(HTML('<h4>Asset Statistics & Optimal Weights</h4>'), asset_styler)
    display(var_styler)
    display(cov_styler)
    print('\n---')
    return {'weights': pd.Series(optimal_weights, index=assets),
            'statistics': port_stats, 'sample_returns': asset_returns,
            'monthly_rf': rf_rate}

# --- 3. Run only when the button is clicked ---
def on_markowitz_run(_):
    with mw_output:
        clear_output(wait=True)
        try:
            start = datetime.date.fromisoformat(mw_start_box.value.strip().replace('.', '-'))
            through = datetime.date.fromisoformat(mw_through_box.value.strip().replace('.', '-'))
        except ValueError:
            print('Invalid date. Use YYYY-MM-DD or YYYY.MM.DD.')
            return
        selected = [ticker for ticker, checkbox in mw_checkboxes.items() if checkbox.value]
        run_markowitz_optimizer(start, through, selected, mw_allow_short.value)

mw_run_button.on_click(on_markowitz_run)
mw_boxes = list(mw_checkboxes.values())
mw_midpoint = len(mw_boxes) // 2
mw_checkbox_grid = widgets.VBox([
    widgets.HBox(mw_boxes[:mw_midpoint]),
    widgets.HBox(mw_boxes[mw_midpoint:])
])
mw_controls = widgets.VBox([
    widgets.HBox([mw_start_box, mw_through_box]),
    widgets.Label('Select Assets:'), mw_checkbox_grid, mw_allow_short, mw_run_button
])
display(widgets.VBox([mw_controls, mw_output]))


---

### Question 1

In the two-fund example, the optimal risky portfolio $P$ is 40% debt fund and 60% equity fund. 

* Investor Alpha and Investor Beta agree on the fund inputs and the 5% risk-free rate, but Alpha is more risk-averse than Beta.

* Do they choose different weights within $P$? How do their complete portfolios differ? 

Explain using the separation property.

---



### Question 2

For an equally weighted portfolio of $N$ assets, write its variance using the average individual variance and the average covariance between different assets. 

Which part shrinks as $N$ grows? Why does diversification reduce firm-specific risk but leave risk shared across assets?

---



### Question 3

Consider Stock Alpha and Stock Beta, plus a risk-free asset:

| Asset | Expected return | Standard deviation |
|:--|--:|--:|
| Stock Alpha | 10% | 15% |
| Stock Beta | 18% | 25% |
| Risk-free asset | 4% | 0% |

The correlation between the two stocks is 0.20. Assume no short selling. Investor Gamma has risk-aversion coefficient $A=5$.

1. Find the weights, expected return, and standard deviation of the optimal risky portfolio $P$.

2. What fraction $y^*$ of Gamma's wealth should be invested in $P$?

3. Find the expected return and standard deviation of Gamma's complete portfolio $C$.

---



### Question 4

Consider two risky assets and a risk-free asset:

| Asset | Expected return | Standard deviation |
|:--|--:|--:|
| Asset Alpha | 8% | 12% |
| Asset Beta | 15% | 22% |
| Risk-free asset | 3% | 0% |

The return correlation between Alpha and Beta is 0.40. Consider a portfolio with 30% in Alpha and 70% in Beta.

1. Calculate this portfolio's expected return, standard deviation, and Sharpe ratio.

2. Find the weights and Sharpe ratio of the optimal risky portfolio. Is the 30/70 portfolio optimal?

---



### Question 5

A manager holds risky portfolio $P$ and considers adding Asset Gamma:

| Investment | Expected return | Standard deviation |
|:--|--:|--:|
| Portfolio $P$ | 12% | 18% |
| Asset Gamma | 9% | 20% |

Their return correlation is 0.10, and the risk-free rate is 4%.

1. Calculate the Sharpe ratio of $P$.
2. For a new risky portfolio with 95% in $P$ and 5% in Gamma, calculate its expected return, standard deviation, and Sharpe ratio.
3. Does this 5% addition improve the risky portfolio? Under the separation property, would the investor's risk aversion change this asset-selection decision?

---



### Derivation of the Two-Asset Tangency Weight

Let $w=w_\alpha$ be the weight in Asset Alpha and $1-w=w_\beta$ the weight in Asset Beta. Define their excess expected returns and covariance as

$$m_\alpha=E(r_\alpha)-r_f,\qquad
m_\beta=E(r_\beta)-r_f,\qquad
c_{\alpha\beta}=\operatorname{Cov}(r_\alpha,r_\beta).$$

The risky portfolio has excess expected return and variance

$$m(w)=m_\beta+w(m_\alpha-m_\beta),$$
$$v(w)=w^2\sigma_\alpha^2+(1-w)^2\sigma_\beta^2+2w(1-w)c_{\alpha\beta}.$$

To maximize its Sharpe ratio $S(w)=m(w)/\sqrt{v(w)}$, differentiate:

$$S'(w)=\frac{2m'(w)v(w)-m(w)v'(w)}{2v(w)^{3/2}},$$
$$m'(w)=m_\alpha-m_\beta,\qquad
\frac{v'(w)}{2}=w\sigma_\alpha^2-(1-w)\sigma_\beta^2+(1-2w)c_{\alpha\beta}.$$

At an interior optimum, $S'(w)=0$, so

$$(m_\alpha-m_\beta)v(w)
=m(w)\left[w\sigma_\alpha^2-(1-w)\sigma_\beta^2+(1-2w)c_{\alpha\beta}\right].$$

Substituting $m(w)$ and $v(w)$ and collecting the terms in $w$ gives

$$w\left[m_\alpha\sigma_\beta^2+m_\beta\sigma_\alpha^2
-(m_\alpha+m_\beta)c_{\alpha\beta}\right]
=m_\alpha\sigma_\beta^2-m_\beta c_{\alpha\beta}.$$

Hence the tangency-weight candidate is

$$w_\alpha^*=
\frac{m_\alpha\sigma_\beta^2-m_\beta c_{\alpha\beta}}
{m_\alpha\sigma_\beta^2+m_\beta\sigma_\alpha^2
-(m_\alpha+m_\beta)c_{\alpha\beta}},
\qquad w_\beta^*=1-w_\alpha^*.$$

With no short selling, compare the Sharpe ratios at $w_\alpha=0$ and $w_\alpha=1$ and, when the denominator is nonzero and $0<w_\alpha^*<1$, at the interior candidate. Choose the highest. In Questions 3 and 4, the interior candidates give the highest Sharpe ratios.

---



In [14]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- Plotting Function ---
def plot_hurdle_rate(sharpe_x):
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Correlation range from -1 to 1
    rho = np.linspace(-1, 1, 100)
    
    # The Hurdle Condition: Sy > Sx * rho
    hurdle_sy = sharpe_x * rho
    
    # Plot the hurdle line
    ax.plot(rho, hurdle_sy, color='navy', linewidth=3, label='Indifference Line (Add if above)')
    
    # Fill areas to show "Add" vs "Don't Add" zones
    ax.fill_between(rho, hurdle_sy, 2, color='green', alpha=0.15, label='Add Asset Y')
    ax.fill_between(rho, -2, hurdle_sy, color='red', alpha=0.1, label='Reject Asset Y')
    
    # Annotations for key correlations
    ax.scatter([1, 0, -0.5], [sharpe_x, 0, -0.5*sharpe_x], color='black', s=80, zorder=5)
    ax.text(1.02, sharpe_x, f'ρ=1\nMust beat\nPortfolio ({sharpe_x})', verticalalignment='center', fontsize=10)
    ax.text(0.02, 0.1, 'ρ=0\nMust just\nbe positive', verticalalignment='bottom', fontsize=10)
    ax.text(-0.5, -0.5*sharpe_x - 0.3, 'ρ=-0.5\nCan have\nneg. Sharpe', horizontalalignment='center', fontsize=10)

    # Formatting
    ax.set_title(f'Hurdle Rate for Adding a New Asset (Portfolio Sharpe = {sharpe_x})', fontsize=14)
    ax.set_xlabel('Correlation with Existing Portfolio (ρ)', fontsize=12)
    ax.set_ylabel('Required Sharpe Ratio of New Asset ($S_Y$)', fontsize=12)
    ax.axhline(0, color='black', linewidth=1)
    ax.axvline(0, color='black', linewidth=1)
    ax.set_xlim(-1.1, 1.3) # Extra space for text
    ax.set_ylim(-1.5, 2.0)
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(loc='upper left', frameon=True)
    
    plt.tight_layout()
    plt.show()

# --- Interactive Setup ---
sharpe_slider = widgets.FloatSlider(
    value=0.5, min=0.1, max=1.5, step=0.1,
    description='Portfolio Sharpe ($S_X$):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='50%'),
    continuous_update=True
)

out = widgets.interactive_output(plot_hurdle_rate, {'sharpe_x': sharpe_slider})

# --- FIX: Use HTMLMath to render LaTeX correctly ---
header = widgets.HTMLMath("<h3>The Diversification Hurdle: $S_Y > S_X \\cdot \\rho_{XY}$</h3>")

display(widgets.VBox([header, sharpe_slider, out]))

### What is next?
* Index Models and The Capital Asset Pricing Model
  * The Single-Index Model
  * The Capital Asset Pricing Model
  * Reading(s): BKM Ch. 8 & 9
  * Assignment #1 (Due before Lecture 11)
  * Suggested Problems
    * Ch. 7: 4-10, 12, 22-27.
    * Ch. 7 – CFA Problems: 1-4, 12.

---
